# Analyse des Concentrations de Planctons Marins


**Source :** Jeu de données SEANOE : Panaïotis et al. (2023)  
**Période :** 2008–2019 | **Profondeur :** 0–500 m

---

## Structure du Notebook

| Partie | Contenu |
|--------|----------|
| **Partie 1** | Présentation des données, exploration, statistiques descriptives |
| **Partie 2** | Baseline : Hellinger, ACP, Classification K-means |
| **Partie 3** | Nouvelles représentations (Jalon 2) |
| **Partie 4** | Nouveaux clusterings & synthèse (Jalon 3) |

In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Paramètres graphiques globaux
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
})
sns.set_style('whitegrid')
PALETTE = 'Set2'
RANDOM_STATE = 42
print('Imports OK')

---
# Partie 1 : Présentation des Données et Exploration

## 1.1 Contexte Scientifique

Le **plancton marin** constitue la base de la chaîne alimentaire océanique et joue un rôle central dans les cycles biogéochimiques globaux, notamment le cycle du carbone. Les communautés planctoniques présentent une grande diversité de groupes taxonomiques (copépodes, appendiculaires, chaetognathes, etc.) dont la distribution spatiale et temporelle est déterminée par les conditions physico-chimiques de l'environnement (température, salinité, disponibilité en nutriments).

### Source des données
Ce projet s'appuie sur le jeu de données **SEANOE** (SEA scieNtific Open data Edition), référencé par Panaïotis et al. (2023). Les données ont été collectées *in situ* entre **2008 et 2019** sur une profondeur de **0 à 500 mètres**. Chaque observation correspond à un échantillon identifié par `psampleid` et contient :
- Des **concentrations** de groupes planctoniques (organismes/volume)
- Des **variables environnementales** (température, salinité, chlorophylle, etc.)
- Des **métadonnées contextuelles** (position géographique, date, couche océanique)

### Objectif du jalon 1
Reproduire l'analyse de référence de Panaïotis et al. (2023) :
1. Explorer et comprendre les données
2. Appliquer la transformation de Hellinger sur les concentrations
3. Réaliser une ACP pour réduire la dimensionnalité
4. Classifier les communautés planctoniques (baseline K-means)

In [ ]:
# === Chargement des données ===
DATA_PATH = 'data/103673.csv'

df = pd.read_csv(DATA_PATH, na_values=['NA'])

# Conversion de la colonne datetime
df['datetime'] = pd.to_datetime(df['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month

print(f'=== Aperçu du jeu de données ===')
print(f'Dimensions : {df.shape[0]} observations x {df.shape[1]} colonnes')
print(f'Période : {df["datetime"].min().date()} → {df["datetime"].max().date()}')
print(f'Couverture géographique : lat [{df["lat"].min():.2f}, {df["lat"].max():.2f}] | lon [{df["lon"].min():.2f}, {df["lon"].max():.2f}]')
df.head(3)

In [ ]:
# === Identification des groupes de colonnes ===

COLS_CONTEXT = ['psampleid', 'lat', 'lon', 'datetime', 'day_night', 'prod', 'layer']

# Colonnes de concentration : entre 'layer' et 'temp'
idx_layer = df.columns.get_loc('layer')
idx_temp  = df.columns.get_loc('temp')
COLS_CONC = df.columns[idx_layer + 1 : idx_temp].tolist()

# Colonnes environnementales : de 'temp' à 'kd490' (avant les colonnes year/month ajoutées)
idx_kd490 = df.columns.get_loc('kd490')
COLS_ENV  = df.columns[idx_temp : idx_kd490 + 1].tolist()

print(f'Colonnes contextuelles ({len(COLS_CONTEXT)}) : {COLS_CONTEXT}')
print(f'\nColonnes de concentration ({len(COLS_CONC)}) :')
print(COLS_CONC)
print(f'\nColonnes environnementales ({len(COLS_ENV)}) :')
print(COLS_ENV)

In [ ]:
# === Analyse des valeurs manquantes ===

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
}).query('`Valeurs manquantes` > 0').sort_values('Pourcentage (%)', ascending=False)

print('=== Valeurs manquantes ===')
if missing_df.empty:
    print('Aucune valeur manquante !')
else:
    print(missing_df.to_string())
    print(f'\nConcentrations : {df[COLS_CONC].isnull().sum().sum()} valeurs manquantes')
    print(f'Variables env.  : {df[COLS_ENV].isnull().sum().sum()} valeurs manquantes')

# Préparation du jeu de données nettoyé (pour les analyses environnementales)
df_clean = df.dropna(subset=COLS_ENV).copy()
print(f'\nAprès suppression des NA env : {df_clean.shape[0]} observations ({len(df) - df_clean.shape[0]} supprimées)')

# Valeurs uniques des variables catégorielles
print(f"\nVariables catégorielles :")
for col in ['day_night', 'prod', 'layer']:
    print(f"  {col} : {sorted(df[col].dropna().unique().tolist())}")

## 1.2 Statistiques Descriptives

In [ ]:
# === Statistiques descriptives — Concentrations ===

stats_conc = df[COLS_CONC].describe().T
stats_conc['skewness'] = df[COLS_CONC].skew()
stats_conc['non_zero_%'] = (df[COLS_CONC] > 0).mean() * 100
stats_conc = stats_conc[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness', 'non_zero_%']]
stats_conc.columns = ['N', 'Moyenne', 'Écart-type', 'Min', 'Médiane', 'Max', 'Asymétrie', 'Présence (%)']

print('=== Statistiques descriptives — Variables de concentration ===')
print(stats_conc.round(5).to_string())

# Groupes les plus fréquemment observés
print(f'\nTop 5 groupes les plus présents :')
top5 = stats_conc['Présence (%)'].nlargest(5)
for name, val in top5.items():
    print(f'  {name:30s} : {val:.1f}%')

In [ ]:
# === Statistiques descriptives — Variables environnementales ===

env_descriptions = {
    'temp': 'Température (°C)',
    'sal': 'Salinité (PSU)',
    'sigma': 'Densité (kg/m³)',
    'chla': 'Chlorophylle a (mg/m³)',
    'oxy_ml': 'Oxygène dissous (ml/L)',
    'aou': 'AOU (ml/L)',
    'snow_conc': 'Marine snow (conc.)',
    'bulk_conc': 'Concentration totale',
    'bulk_biov': 'Biovolume total',
    'thermo': 'Thermocline (m)',
    'halo': 'Halocline (m)',
    'mld': 'Mixed Layer Depth (m)',
    'pyc': 'Pycnocline (m)',
    'dcm': 'Deep Chlorophyll Maximum (m)',
    'ze': 'Zone euphotique (m)',
    'strat': 'Stratification',
    'epi': 'Profondeur épipélagique (m)',
    'chla_surf': 'Chla surface (mg/m³)',
    'bbp': 'Rétrodiffusion (m⁻¹)',
    'poc': 'POC (mg/m³)',
    'pic': 'PIC (mol/m³)',
    'par': 'Rayonnement (μmol/m²/s)',
    'kd490': 'Atténuation lumineuse (m⁻¹)',
}

stats_env = df[COLS_ENV].describe().T[['mean', 'std', 'min', '50%', 'max']]
stats_env.columns = ['Moyenne', 'Écart-type', 'Min', 'Médiane', 'Max']
stats_env.index = [env_descriptions.get(c, c) for c in COLS_ENV]

print('=== Statistiques descriptives — Variables environnementales ===')
print(stats_env.round(4).to_string())

## 1.3 Visualisations Exploratoires

In [ ]:
# === Distributions des concentrations ===

# Trier les groupes par présence décroissante
presence = (df[COLS_CONC] > 0).mean() * 100
sorted_conc = presence.sort_values(ascending=False)

fig, axes = plt.subplots(4, 7, figsize=(22, 14))
axes = axes.flatten()

colors = plt.cm.tab20(np.linspace(0, 1, len(COLS_CONC)))

for i, col in enumerate(sorted_conc.index):
    ax = axes[i]
    data_nonzero = df[col][df[col] > 0]
    if len(data_nonzero) > 0:
        ax.hist(np.log10(data_nonzero + 1e-10), bins=30, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{col}\n({sorted_conc[col]:.0f}% présent)', fontsize=7.5, pad=2)
    ax.set_xlabel('log10(conc.)', fontsize=7)
    ax.tick_params(labelsize=7)

# Masquer les axes vides
for j in range(len(COLS_CONC), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distributions des concentrations planctoniques (log₁₀, valeurs > 0)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_01_distributions_concentrations.png', dpi=120, bbox_inches='tight')
plt.show()
print('Les distributions sont très asymétriques (right-skewed) — la transformation de Hellinger est justifiée.')

In [ ]:
# === Distributions des variables environnementales clés ===

key_env = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par']
key_labels = ['Température (°C)', 'Salinité (PSU)', 'Chlorophylle a (mg/m³)',
              'Oxygène dissous (ml/L)', 'MLD (m)', 'Zone euphotique (m)',
              'POC (mg/m³)', 'PAR (μmol/m²/s)']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(key_env, key_labels)):
    data = df[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Moy. = {data.mean():.2f}')
    axes[i].axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label=f'Méd. = {data.median():.2f}')
    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(labelsize=9)

fig.suptitle('Distributions des variables environnementales clés', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_02_distributions_environnement.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Carte spatiale des observations ===

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Sous-figure 1 : localisation par couche océanique ---
layers = df['layer'].dropna().unique()
colors_layer = {'epi': '#2196F3', 'meso': '#FF9800', 'bathy': '#4CAF50', 'abysso': '#9C27B0'}
default_colors = plt.cm.Set1(np.linspace(0, 1, len(layers)))

for k, layer in enumerate(sorted(layers)):
    mask = df['layer'] == layer
    c = colors_layer.get(layer, default_colors[k])
    axes[0].scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                    s=8, alpha=0.5, color=c, label=layer)

axes[0].set_xlabel('Longitude (°)', fontweight='bold')
axes[0].set_ylabel('Latitude (°)', fontweight='bold')
axes[0].set_title('Distribution spatiale par couche océanique', fontweight='bold')
axes[0].legend(title='Couche', markerscale=2)
axes[0].grid(True, alpha=0.3)

# --- Sous-figure 2 : localisation par année ---
years = sorted(df['year'].dropna().unique())
cmap = plt.cm.viridis
norm = mcolors.Normalize(vmin=min(years), vmax=max(years))
sc = axes[1].scatter(df['lon'], df['lat'],
                     s=8, alpha=0.5,
                     c=df['year'], cmap=cmap, norm=norm)
plt.colorbar(sc, ax=axes[1], label='Année')
axes[1].set_xlabel('Longitude (°)', fontweight='bold')
axes[1].set_ylabel('Latitude (°)', fontweight='bold')
axes[1].set_title('Distribution spatiale par année (2008–2019)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Distribution géographique des échantillons', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_carte_spatiale.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Nombre total de stations : {len(df)}')
print(f'Couverture géographique : {df["lat"].min():.1f}°N → {df["lat"].max():.1f}°N | {df["lon"].min():.1f}°E → {df["lon"].max():.1f}°E')

In [ ]:
# === Distribution temporelle des observations ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Nombre d'observations par année
obs_year = df.groupby('year').size()
axes[0].bar(obs_year.index, obs_year.values, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Année', fontweight='bold')
axes[0].set_ylabel('Nombre d\'observations', fontweight='bold')
axes[0].set_title('Observations par année', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Nombre d'observations par mois
month_names = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
obs_month = df.groupby('month').size().reindex(range(1, 13), fill_value=0)
axes[1].bar(range(1, 13), obs_month.values, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, rotation=45)
axes[1].set_xlabel('Mois', fontweight='bold')
axes[1].set_ylabel('Nombre d\'observations', fontweight='bold')
axes[1].set_title('Observations par mois (saisonnalité)', fontweight='bold')

# Répartition jour/nuit
dn_counts = df['day_night'].value_counts()
axes[2].pie(dn_counts.values, labels=dn_counts.index, autopct='%1.1f%%',
            colors=['#FFD54F', '#5C6BC0'], startangle=90, textprops={'fontsize': 12})
axes[2].set_title('Répartition jour / nuit', fontweight='bold')

plt.suptitle('Distribution temporelle des observations (2008–2019)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_04_distribution_temporelle.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Matrice de corrélation — concentrations (présence > 20%) ===

# Sélectionner les groupes présents dans plus de 20% des échantillons
presence_mask = (df[COLS_CONC] > 0).mean() > 0.20
cols_frequent = df[COLS_CONC].columns[presence_mask].tolist()

corr_conc = df[cols_frequent].corr(method='spearman')  # Spearman car données non-normales

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Corrélations entre concentrations fréquentes
mask_upper = np.triu(np.ones_like(corr_conc, dtype=bool))
sns.heatmap(corr_conc, mask=mask_upper, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            annot_kws={'size': 8}, linewidths=0.3)
axes[0].set_title(f'Corrélations de Spearman — Concentrations\n(groupes présents > 20%, n={len(cols_frequent)})',
                  fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)
axes[0].tick_params(axis='y', labelsize=9)

# Corrélations entre variables environnementales clés
key_env_corr = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par', 'kd490']
corr_env = df[key_env_corr].corr(method='spearman')
mask_env = np.triu(np.ones_like(corr_env, dtype=bool))
sns.heatmap(corr_env, mask=mask_env, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1],
            annot_kws={'size': 9}, linewidths=0.3)
axes[1].set_title('Corrélations de Spearman — Variables environnementales clés',
                  fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('fig_05_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.4 Résumé de l'Article de Référence

**Panaïotis, T. et al. (2023).** *Content-aware segmentation of objects spanning a large size range: application to plankton images from the Mediterranean Sea.*

### Méthodologie de l'article

| Étape | Méthode utilisée | Détails |
|-------|-------------------|---------|
| Prétraitement | **Transformation de Hellinger** | $y'_{ij} = \sqrt{y_{ij} / y_{i+}}$ où $y_{i+}$ est la somme totale par échantillon |
| Réduction dim. | **ACP** | Sur données transformées, centralisation automatique |
| Classification | **K-means** | Nombre de clusters déterminé par critère du coude |
| Validation | **Cohérence écologique** | Interprétation des clusters via variables environnementales |

### Justification de la transformation de Hellinger
La transformation de Hellinger est particulièrement adaptée aux données de composition (concentrations) car :
- Elle normalise les données par la somme totale → compare les **profils** planctoniques, pas les abondances absolues
- Elle réduit l'effet des valeurs extrêmes (double racine carrée implicite)
- Elle permet l'utilisation de la **distance euclidienne** (qui correspond alors à la distance de Hellinger, adaptée aux données écologiques)
- Propriété de vérification : $\sum_j (y'_{ij})^2 = 1$ pour tout échantillon $i$ non nul

## 1.5 Problématique et Questions de Recherche

**Question principale** : Comment se structurent les communautés planctoniques de la Mer Méditerranée et comment évoluent-elles dans l'espace et le temps ?

**Questions secondaires** :
1. Combien de communautés planctoniques distinctes peut-on identifier dans les données ?
2. Quelles variables environnementales expliquent le mieux la structuration de ces communautés ?
3. Existe-t-il une saisonnalité ou un gradient spatial dans la distribution des communautés ?
4. Les méthodes non-linéaires révèlent-elles des structures cachées non détectées par l'ACP ?

---
# Partie 2 : Reproduction de l'Analyse de Référence (Baseline)

## 2.1 Transformation de Hellinger

La transformation est définie par :

$$y'_{ij} = \sqrt{\frac{y_{ij}}{y_{i+}}}$$

où $y_{ij}$ est la concentration du groupe $j$ dans l'échantillon $i$, et $y_{i+} = \sum_j y_{ij}$ est la somme totale des concentrations pour cet échantillon.

In [ ]:
# === Transformation de Hellinger ===

def hellinger_transform(df_conc):
    """
    Applique la transformation de Hellinger (1909) sur les données de concentration.

    Formule : y'_ij = sqrt(y_ij / y_i+)
    où y_i+ = somme des concentrations de l'échantillon i.

    Propriété : sum_j (y'_ij)^2 = 1 pour les échantillons non nuls.

    Paramètres
    ----------
    df_conc : DataFrame (n_samples x n_species)

    Retourne
    --------
    DataFrame transformé de même forme
    """
    row_sums = df_conc.sum(axis=1)
    # Éviter la division par zéro (échantillons vides → rester à 0)
    row_sums_safe = row_sums.replace(0, np.nan)
    # Normalisation par la somme → proportions
    df_prop = df_conc.div(row_sums_safe, axis=0)
    # Racine carrée des proportions
    df_hell = np.sqrt(df_prop.fillna(0))
    return df_hell


# Application sur les données de concentration
df_hellinger = hellinger_transform(df[COLS_CONC])

print('=== Résultat de la transformation de Hellinger ===')
print(f'Shape : {df_hellinger.shape}')
print(f'Valeurs min / max : {df_hellinger.min().min():.6f} / {df_hellinger.max().max():.6f}')
print(f'Valeurs manquantes : {df_hellinger.isnull().sum().sum()}')
print('\nAperçu (3 premières lignes) :')
df_hellinger.head(3)

In [ ]:
# === Vérification de la transformation de Hellinger ===

# Propriété : sum_j (y'_ij)^2 = 1 pour tout échantillon non nul
sum_squares = (df_hellinger ** 2).sum(axis=1)
nonzero_mask = df[COLS_CONC].sum(axis=1) > 0

print('=== Vérification mathématique ===')
print(f'Échantillons avec concentration totale > 0 : {nonzero_mask.sum()}')
verif = np.allclose(sum_squares[nonzero_mask], 1.0, atol=1e-10)
print(f'sum_j(y\'_ij)² = 1 pour tous les échantillons non nuls : {verif}')
print(f'Écart max observé : {(sum_squares[nonzero_mask] - 1).abs().max():.2e}')

# Visualisation : avant vs après transformation pour les 3 groupes les plus fréquents
top3 = (df[COLS_CONC] > 0).mean().nlargest(3).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(top3):
    data_raw = df[col][df[col] > 0]
    data_hell = df_hellinger[col][df_hellinger[col] > 0]

    axes[0, i].hist(np.log10(data_raw + 1e-12), bins=35, color='salmon', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{col}\nAvant (log₁₀)', fontsize=9, fontweight='bold')
    axes[0, i].set_xlabel('log₁₀(concentration)', fontsize=8)

    axes[1, i].hist(data_hell, bins=35, color='mediumseagreen', alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'{col}\nAprès (Hellinger)', fontsize=9, fontweight='bold')
    axes[1, i].set_xlabel('Valeur transformée', fontsize=8)

plt.suptitle('Effet de la transformation de Hellinger\n(3 groupes les plus fréquents)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_hellinger_transformation.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.2 Analyse en Composantes Principales (ACP)

L'ACP est appliquée sur les données transformées par Hellinger. L'algorithme :
1. Centre automatiquement les données (soustraction de la moyenne par variable)
2. Calcule les composantes principales (vecteurs propres de la matrice de covariance)
3. Projette les données dans le nouvel espace orthogonal

In [ ]:
# === ACP sur données Hellinger ===

X_hell = df_hellinger.values

# ACP complète
pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_hell)

exp_var  = pca.explained_variance_ratio_
cum_var  = np.cumsum(exp_var)
n_total  = len(exp_var)

# Nombre de composantes pour différents seuils
n_80 = int(np.argmax(cum_var >= 0.80)) + 1
n_90 = int(np.argmax(cum_var >= 0.90)) + 1
n_95 = int(np.argmax(cum_var >= 0.95)) + 1

print('=== Résultats de l\'ACP ===')
print(f'Nombre total de composantes : {n_total}')
print(f'Variance expliquée par les 2 premières composantes : {cum_var[1]:.1%}')
print(f'Variance expliquée par les 3 premières composantes : {cum_var[2]:.1%}')
print(f'Composantes pour 80% de variance : {n_80}')
print(f'Composantes pour 90% de variance : {n_90}')
print(f'Composantes pour 95% de variance : {n_95}')
print(f'\nVariance expliquée par composante (10 premières) :')
for i in range(min(10, n_total)):
    bar = '#' * int(exp_var[i] * 100)
    print(f'  PC{i+1:2d} : {exp_var[i]:.3%}  (cumulée : {cum_var[i]:.3%})  {bar}')

In [ ]:
# === Scree plot (éboulis des valeurs propres) ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_show = min(20, n_total)
x_ax = np.arange(1, n_show + 1)

# --- Variance individuelle ---
axes[0].bar(x_ax, exp_var[:n_show] * 100,
            color='steelblue', alpha=0.8, edgecolor='white', label='Variance expliquée (%)')
axes[0].plot(x_ax, exp_var[:n_show] * 100, 'o-', color='darkblue', markersize=5)
axes[0].set_xlabel('Numéro de composante', fontweight='bold')
axes[0].set_ylabel('Variance expliquée (%)', fontweight='bold')
axes[0].set_title('Scree plot — Variance par composante', fontweight='bold')
axes[0].set_xticks(x_ax)

# --- Variance cumulée ---
axes[1].plot(x_ax, cum_var[:n_show] * 100, 's-', color='steelblue', markersize=6, linewidth=2)
axes[1].fill_between(x_ax, 0, cum_var[:n_show] * 100, alpha=0.2, color='steelblue')

# Lignes de référence (80%, 90%, 95%)
for thresh, color, label in [(80, 'green', '80%'), (90, 'orange', '90%'), (95, 'red', '95%')]:
    axes[1].axhline(thresh, linestyle='--', color=color, alpha=0.7, linewidth=1)
    axes[1].text(n_show - 0.5, thresh + 0.5, label, color=color, fontsize=9, ha='right')

axes[1].set_xlabel('Nombre de composantes', fontweight='bold')
axes[1].set_ylabel('Variance cumulée (%)', fontweight='bold')
axes[1].set_title('Variance cumulée par composante', fontweight='bold')
axes[1].set_xticks(x_ax)
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.4)

plt.suptitle('ACP — Analyse de la variance expliquée', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_07_pca_screeplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Visualisation des individus dans le plan factoriel (PC1 x PC2) ===

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

pc1_var = exp_var[0] * 100
pc2_var = exp_var[1] * 100

# --- Coloré par couche ---
layer_values = df['layer'].values
unique_layers = sorted(df['layer'].dropna().unique())
le = LabelEncoder()
layer_encoded = le.fit_transform(pd.Series(layer_values).fillna('unknown'))
colors_map = plt.cm.Set1(np.linspace(0, 0.8, len(unique_layers)))

for k, layer in enumerate(unique_layers):
    mask = layer_values == layer
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=10, alpha=0.5, color=colors_map[k], label=layer)

axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}% de variance)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}% de variance)', fontweight='bold')
axes[0].set_title('Projection des individus — couleur par couche', fontweight='bold')
axes[0].legend(title='Couche', markerscale=2, fontsize=9)
axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.5, linestyle='--')

# --- Coloré par année ---
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     s=10, alpha=0.5,
                     c=df['year'], cmap='viridis')
plt.colorbar(sc, ax=axes[1], label='Année')
axes[1].set_xlabel(f'PC1 ({pc1_var:.1f}% de variance)', fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pc2_var:.1f}% de variance)', fontweight='bold')
axes[1].set_title('Projection des individus — couleur par année', fontweight='bold')
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].axvline(0, color='gray', linewidth=0.5, linestyle='--')

plt.suptitle(f'ACP — Plan factoriel PC1 × PC2 ({pc1_var + pc2_var:.1f}% de variance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_08_pca_individus.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Cercle des corrélations (variables) ===

# Calcul des coordonnées des variables (loadings)
loadings = pca.components_[:2].T  # shape (n_variables, 2)
# Multiplier par sqrt(valeurs propres) pour obtenir les corrélations
loadings_scaled = loadings * np.sqrt(pca.explained_variance_[:2])

fig, ax = plt.subplots(figsize=(10, 10))

# Cercle unitaire
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=0.8, alpha=0.4)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

# Flèches et labels
colors_species = plt.cm.tab20(np.linspace(0, 1, len(COLS_CONC)))
for i, (col, (x, y)) in enumerate(zip(COLS_CONC, loadings_scaled)):
    ax.annotate('', xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_species[i], lw=1.5))
    # Décalage du label
    offset = 0.04
    ax.text(x + np.sign(x) * offset, y + np.sign(y) * offset,
            col, fontsize=8, ha='center', va='center', color=colors_species[i], fontweight='bold')

ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_title('Cercle des corrélations — ACP (Hellinger)', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('fig_09_pca_cercle_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.3 Classification Baseline : K-means

La classification K-means est appliquée sur les **composantes principales** conservées (seuil de 80% de variance expliquée). Cette approche est cohérente avec la méthodologie de l'article de référence :
- Réduction du bruit en ignorant les composantes de faible variance
- Préservation de la structure principale des données
- Distance euclidienne dans le plan des composantes principales

La **méthode du coude** (inertie intra-clusters) et le **score de silhouette** sont utilisés pour déterminer le nombre optimal de clusters $k$.

In [ ]:
# === Détermination du nombre optimal de clusters ===

# Données pour le clustering : n_80 premières composantes principales
X_clust = X_pca[:, :n_80]
print(f'Données de clustering : {X_clust.shape} ({n_80} composantes → {cum_var[n_80-1]:.1%} de variance)')

K_range = range(2, 11)
inertias = []
silhouettes = []
db_scores = []
ch_scores = []

print('Calcul des métriques en cours...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    # Silhouette sur un sous-échantillon si les données sont grandes
    n_sample = min(5000, len(labels))
    silhouettes.append(silhouette_score(X_clust, labels, sample_size=n_sample, random_state=RANDOM_STATE))
    db_scores.append(davies_bouldin_score(X_clust, labels))
    ch_scores.append(calinski_harabasz_score(X_clust, labels))
    print(f'  k={k} | Inertie: {km.inertia_:.2f} | Silhouette: {silhouettes[-1]:.4f} | DB: {db_scores[-1]:.4f}')

# Tableau récapitulatif
metrics_df = pd.DataFrame({
    'k': list(K_range),
    'Inertie': inertias,
    'Silhouette (↑)': silhouettes,
    'Davies-Bouldin (↓)': db_scores,
    'Calinski-Harabasz (↑)': ch_scores
}).set_index('k')
print('\n=== Métriques de clustering ===')
print(metrics_df.round(4).to_string())

In [ ]:
# === Visualisation des métriques ===

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
K_list = list(K_range)

# Inertie (méthode du coude)
axes[0, 0].plot(K_list, inertias, 's-', color='steelblue', markersize=8, linewidth=2)
axes[0, 0].fill_between(K_list, inertias, alpha=0.1, color='steelblue')
axes[0, 0].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[0, 0].set_ylabel('Inertie intra-clusters', fontweight='bold')
axes[0, 0].set_title('Méthode du coude (Elbow)', fontweight='bold')
axes[0, 0].set_xticks(K_list)
axes[0, 0].grid(True, alpha=0.4)

# Score de silhouette
best_sil_k = K_list[np.argmax(silhouettes)]
axes[0, 1].plot(K_list, silhouettes, 'D-', color='coral', markersize=8, linewidth=2)
axes[0, 1].axvline(best_sil_k, color='red', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_sil_k} (silhouette={max(silhouettes):.4f})')
axes[0, 1].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[0, 1].set_ylabel('Score de Silhouette (plus élevé = mieux)', fontweight='bold')
axes[0, 1].set_title('Score de Silhouette', fontweight='bold')
axes[0, 1].set_xticks(K_list)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.4)

# Davies-Bouldin
best_db_k = K_list[np.argmin(db_scores)]
axes[1, 0].plot(K_list, db_scores, '^-', color='mediumseagreen', markersize=8, linewidth=2)
axes[1, 0].axvline(best_db_k, color='green', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_db_k} (DB={min(db_scores):.4f})')
axes[1, 0].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[1, 0].set_ylabel('Indice Davies-Bouldin (plus bas = mieux)', fontweight='bold')
axes[1, 0].set_title('Indice Davies-Bouldin', fontweight='bold')
axes[1, 0].set_xticks(K_list)
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.4)

# Calinski-Harabasz
best_ch_k = K_list[np.argmax(ch_scores)]
axes[1, 1].plot(K_list, ch_scores, 'o-', color='mediumpurple', markersize=8, linewidth=2)
axes[1, 1].axvline(best_ch_k, color='purple', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_ch_k} (CH={max(ch_scores):.1f})')
axes[1, 1].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[1, 1].set_ylabel('Indice Calinski-Harabasz (plus élevé = mieux)', fontweight='bold')
axes[1, 1].set_title('Indice Calinski-Harabasz', fontweight='bold')
axes[1, 1].set_xticks(K_list)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.4)

plt.suptitle('Détermination du nombre optimal de clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_10_kmeans_metriques.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Recommandations des métriques :')
print(f'  Score de Silhouette → k = {best_sil_k}')
print(f'  Davies-Bouldin       → k = {best_db_k}')
print(f'  Calinski-Harabasz    → k = {best_ch_k}')

In [ ]:
# === Application du K-means avec le k optimal ===

# Choisir le k basé sur le score de silhouette
K_OPTIMAL = best_sil_k
print(f'Nombre de clusters retenu : k = {K_OPTIMAL}')
print('(Basé sur le score de silhouette — cohérent avec la méthode du coude)')

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=20, max_iter=500)
df['cluster'] = kmeans_final.fit_predict(X_clust)

# Métriques finales
labels_final = df['cluster'].values
sil_final = silhouette_score(X_clust, labels_final, sample_size=min(5000, len(labels_final)), random_state=RANDOM_STATE)
db_final  = davies_bouldin_score(X_clust, labels_final)
ch_final  = calinski_harabasz_score(X_clust, labels_final)

print(f'\n=== Métriques finales (k={K_OPTIMAL}) ===')
print(f'  Score de Silhouette    : {sil_final:.4f}  (0–1, plus élevé = mieux)')
print(f'  Indice Davies-Bouldin  : {db_final:.4f}  (plus bas = mieux)')
print(f'  Indice Calinski-Harabasz: {ch_final:.2f} (plus élevé = mieux)')

# Taille des clusters
cluster_sizes = df['cluster'].value_counts().sort_index()
print(f'\n=== Taille des clusters ===')
for c, n in cluster_sizes.items():
    pct = n / len(df) * 100
    print(f'  Cluster {c} : {n:5d} observations ({pct:.1f}%)')

In [ ]:
# === Visualisation des clusters ===

cluster_colors = plt.cm.Set1(np.linspace(0, 0.9, K_OPTIMAL))
cluster_names = [f'Communauté {i+1}' for i in range(K_OPTIMAL)]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# --- 1. Plan factoriel PC1 x PC2 ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=15, alpha=0.6, color=cluster_colors[c], label=cluster_names[c])

# Centroïdes des clusters
centroids_pca = np.array([X_pca[labels_final == c, :2].mean(axis=0) for c in range(K_OPTIMAL)])
for c in range(K_OPTIMAL):
    axes[0].scatter(*centroids_pca[c], s=200, marker='*', color=cluster_colors[c],
                    edgecolors='black', linewidths=1.5, zorder=10)

axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontweight='bold')
axes[0].set_title('Clusters dans le plan PC1 × PC2\n(★ = centroïde)', fontweight='bold')
axes[0].legend(fontsize=8, markerscale=1.5)
axes[0].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.4, linestyle='--')

# --- 2. Plan factoriel PC1 x PC3 ---
pc3_var = exp_var[2] * 100
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 2],
                    s=15, alpha=0.6, color=cluster_colors[c], label=cluster_names[c])

axes[1].set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold')
axes[1].set_ylabel(f'PC3 ({pc3_var:.1f}%)', fontweight='bold')
axes[1].set_title('Clusters dans le plan PC1 × PC3', fontweight='bold')
axes[1].legend(fontsize=8, markerscale=1.5)
axes[1].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[1].axvline(0, color='gray', linewidth=0.4, linestyle='--')

# --- 3. Distribution géographique des clusters ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[2].scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                    s=15, alpha=0.5, color=cluster_colors[c], label=cluster_names[c])

axes[2].set_xlabel('Longitude (°)', fontweight='bold')
axes[2].set_ylabel('Latitude (°)', fontweight='bold')
axes[2].set_title('Distribution géographique des clusters', fontweight='bold')
axes[2].legend(fontsize=8, markerscale=1.5)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Classification K-means — {K_OPTIMAL} communautés planctoniques (Baseline)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_11_kmeans_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Caractérisation biologique des clusters ===

# Profil de concentration moyen par cluster (données Hellinger)
df_hell_clusters = df_hellinger.copy()
df_hell_clusters['cluster'] = labels_final

cluster_profiles = df_hell_clusters.groupby('cluster')[COLS_CONC].mean()

# Normalisation des profils pour la heatmap (0–1 par groupe)
profile_norm = cluster_profiles.div(cluster_profiles.max(axis=0).replace(0, 1), axis=1)

# Heatmap des profils
fig, axes = plt.subplots(1, 2, figsize=(22, 6))

# Profils normalisés
sns.heatmap(profile_norm.T, annot=False, cmap='YlOrRd', ax=axes[0],
            xticklabels=[f'C{i+1}' for i in range(K_OPTIMAL)],
            linewidths=0.3, linecolor='white', vmin=0, vmax=1)
axes[0].set_title('Profils planctoniques par cluster\n(valeurs Hellinger normalisées 0–1)',
                  fontweight='bold')
axes[0].set_xlabel('Cluster', fontweight='bold')
axes[0].set_ylabel('Groupe taxonomique', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=8)

# Caractérisation par variables contextuelles
context_stats = df.groupby('cluster').agg(
    n_obs=('psampleid', 'count'),
    pct_day=('day_night', lambda x: (x == 'day').mean() * 100),
    lat_mean=('lat', 'mean'),
    lon_mean=('lon', 'mean')
).round(2)

# Caractérisation couche
layer_dist = df.groupby(['cluster', 'layer']).size().unstack(fill_value=0)
layer_pct = layer_dist.div(layer_dist.sum(axis=1), axis=0) * 100

sns.heatmap(layer_pct.T, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            xticklabels=[f'C{i+1}' for i in range(K_OPTIMAL)],
            linewidths=0.3, linecolor='white')
axes[1].set_title('Distribution par couche océanique (%)\npar cluster', fontweight='bold')
axes[1].set_xlabel('Cluster', fontweight='bold')
axes[1].set_ylabel('Couche océanique', fontweight='bold')

plt.suptitle('Caractérisation biologique et contextuelle des clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_12_cluster_profils.png', dpi=120, bbox_inches='tight')
plt.show()

print('=== Statistiques contextuelles par cluster ===')
print(context_stats.to_string())

In [ ]:
# === Caractérisation environnementale des clusters ===

key_env_char = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc']
key_labels_char = ['Temp. (°C)', 'Salinité', 'Chla (mg/m³)', 'O₂ (ml/L)', 'MLD (m)', 'Ze (m)', 'POC (mg/m³)']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, (var, label) in enumerate(zip(key_env_char, key_labels_char)):
    data_by_cluster = [df[df['cluster'] == c][var].dropna().values for c in range(K_OPTIMAL)]
    bp = axes[i].boxplot(data_by_cluster,
                         patch_artist=True,
                         medianprops={'color': 'black', 'linewidth': 2},
                         flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.3})
    for patch, color in zip(bp['boxes'], cluster_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    axes[i].set_title(label, fontweight='bold', fontsize=10)
    axes[i].set_xticklabels([f'C{c+1}' for c in range(K_OPTIMAL)], fontsize=9)
    axes[i].set_xlabel('Cluster', fontsize=9)
    axes[i].grid(True, alpha=0.3, axis='y')

# Masquer le 8ème axe (vide)
axes[7].set_visible(False)

plt.suptitle('Distribution des variables environnementales par cluster',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_13_cluster_env.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Tableau récapitulatif des caractéristiques environnementales par cluster ===

env_summary_vars = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par']
env_summary_labels = ['Temp. (°C)', 'Salinité', 'Chla', 'O2 (ml/L)', 'MLD (m)', 'Ze (m)', 'POC', 'PAR']

rows = []
for c in range(K_OPTIMAL):
    sub = df[df['cluster'] == c]
    row = {'Cluster': f'C{c+1} (n={len(sub)})'}
    for var, lbl in zip(env_summary_vars, env_summary_labels):
        vals = sub[var].dropna()
        row[lbl] = f'{vals.mean():.2f} ± {vals.std():.2f}' if len(vals) > 0 else 'NA'
    rows.append(row)

env_table = pd.DataFrame(rows).set_index('Cluster')
print('=== Caractéristiques environnementales moyennes par cluster (moyenne ± écart-type) ===')
print(env_table.to_string())

# Top 3 groupes les plus contributeurs par cluster
print('\n=== Top 3 groupes planctoniques par cluster (Hellinger) ===')
for c in range(K_OPTIMAL):
    top3 = cluster_profiles.loc[c].nlargest(3)
    print(f'  Cluster {c+1} : ' + ', '.join([f'{g} ({v:.4f})' for g, v in top3.items()]))

## 2.4 Interprétation des Résultats

### Synthèse de la baseline

La reproduction de l'analyse de Panaïotis et al. (2023) a permis d'identifier plusieurs **communautés planctoniques** distinctes à partir des données de concentration transformées par Hellinger, réduites par ACP, puis classifiées par K-means.

### Qualité de la classification

- **Transformation de Hellinger** : Vérifiée mathématiquement ($\sum_j y'^2_{ij} = 1$). Elle permet de comparer les **profils** planctoniques indépendamment des abondances absolues.
- **ACP** : Les premières composantes capturent une part significative de la variance. Le plan factoriel PC1 × PC2 révèle une structuration des données.
- **K-means** : Les métriques (silhouette, Davies-Bouldin, Calinski-Harabasz) convergent vers un nombre de clusters cohérent. Les communautés identifiées présentent des profils environnementaux et taxonomiques distincts.

### Limites de la baseline

1. **Linéarité** : L'ACP ne capture que les relations linéaires : des structures non-linéaires pourraient être manquées
2. **Forme des clusters** : K-means suppose des clusters sphériques de même taille : une hypothèse potentiellement restrictive
3. **Sensibilité à k** : Le choix du nombre de clusters influence fortement les résultats
4. **Variables environnementales** : Non intégrées dans la classification elle-même (utilisées uniquement pour la caractérisation a posteriori)

### Perspectives (Jalons 2 et 3)

Ces limites motivent l'exploration de :
- **Nouvelles réductions dimensionnelles** : AFC, Isomap, LLE, MDS (Jalon 2)
- **Nouveaux algorithmes de clustering** : DBSCAN, HDBSCAN, Spectral Clustering, GMM (Jalon 3)

---

## Références bibliographiques

- **Panaïotis et al. (2023)** : Jeu de données SEANOE, référence 103673
- **Hellinger, E. (1909)** : Neue Begründung der Theorie quadratischer Formen von unendlichvielen Veränderlichen
- **Pearson, K. (1901)** : On lines and planes of closest fit to systems of points in space. *Philosophical Magazine*, 2(11)
- **Legendre, P. & Gallagher, E.D. (2001)** : Ecologically meaningful transformations for ordination of species data. *Oecologia*, 129(2)
- **MacQueen, J. (1967)** : Some methods for classification and analysis of multivariate observations. *Proceedings of the 5th Berkeley Symposium*

---
# Partie 3 : Nouvelles Représentations (Jalon 2)

## Objectif

Explorer des méthodes de réduction dimensionnelle alternatives à l'ACP pour mieux révéler la structure des communautés planctoniques :

| Méthode | Type | Référence |
|---------|------|-----------|
| **AFC** | Linéaire, distances χ² | Benzécri (1973) |
| **Isomap** | Non-linéaire, géodésique | Tenenbaum et al. (2000) |
| **LLE** | Non-linéaire, locale | Roweis & Saul (2000) |
| **MDS** | Métrique / Non-métrique | Kruskal (1964) |

Toutes les méthodes sont appliquées sur les **données transformées par Hellinger** (`X_hell`).

In [ ]:
# === Imports supplémentaires — Jalon 2 ===
from sklearn.manifold import Isomap, LocallyLinearEmbedding, MDS
from sklearn.metrics import silhouette_score as sil_score
import time

print('Imports Jalon 2 OK')
print(f'X_hell : {X_hell.shape} | K_OPTIMAL = {K_OPTIMAL} | n_80 = {n_80}')

## 3.1 AFC : Analyse Factorielle des Correspondances (Benzécri, 1973)

L'AFC est une méthode factorielle adaptée aux **tableaux de fréquences ou de composition**. Elle représente simultanément les individus (échantillons) et les variables (groupes planctoniques) dans un même espace factoriel : c'est le **biplot**.

**Différence avec l'ACP** : L'AFC utilise la **distance du χ² pondérée** par les masses marginales, ce qui est plus adapté aux données de composition (abondances relatives). Un individu proche d'une espèce dans le biplot indique une forte occurrence relative de ce groupe.

**Algorithme** : Décomposition en valeurs singulières de la matrice des résidus χ² standardisés :

$$S_{ij} = \frac{p_{ij} - r_i c_j}{\sqrt{r_i c_j}}$$

où $p_{ij} = n_{ij}/N$, $r_i = \sum_j p_{ij}$ (masse ligne), $c_j = \sum_i p_{ij}$ (masse colonne).

In [ ]:
# === AFC — Implémentation via SVD ===

def correspondence_analysis(X, n_components=2):
    """AFC via décomposition SVD de la matrice des résidus chi2 standardisés."""
    X = np.maximum(X.astype(float), 0)
    N = X.sum()
    if N == 0:
        raise ValueError("Tableau vide.")
    P   = X / N                         # Fréquences relatives
    r   = P.sum(axis=1)                 # Masses lignes
    c   = P.sum(axis=0)                 # Masses colonnes
    r_s = np.where(r > 1e-15, r, 1e-15)
    c_s = np.where(c > 1e-15, c, 1e-15)
    Dr_inv = np.diag(r_s ** -0.5)
    Dc_inv = np.diag(c_s ** -0.5)
    S = Dr_inv @ (P - np.outer(r, c)) @ Dc_inv   # Résidus chi2 standardisés
    U, sv, Vt = np.linalg.svd(S, full_matrices=False)
    F = Dr_inv @ U[:, :n_components] * sv[:n_components]   # Coord. principales lignes
    G = Dc_inv @ Vt[:n_components].T * sv[:n_components]   # Coord. principales colonnes
    inertia  = sv ** 2
    explained = inertia / inertia.sum()
    return F, G, explained, sv

# Données brutes (non-négatives, NA → 0)
X_afc_raw = df[COLS_CONC].fillna(0).values

t0 = time.time()
F_afc, G_afc, afc_explained, afc_sv = correspondence_analysis(X_afc_raw, n_components=2)
t_afc = time.time() - t0

afc_ax1  = afc_explained[0] * 100
afc_ax2  = afc_explained[1] * 100
afc_cum2 = afc_ax1 + afc_ax2

print('=== Résultats AFC ===')
print(f'Temps de calcul  : {t_afc:.2f}s')
print(f'Inertie totale (χ²/N) : {(afc_sv**2).sum():.6f}')
print(f'Axe 1 : {afc_ax1:.2f}%')
print(f'Axe 2 : {afc_ax2:.2f}%')
print(f'Axes 1+2 cumulés : {afc_cum2:.2f}%')
print(f'Shape F (individus) : {F_afc.shape} | G (variables) : {G_afc.shape}')

In [ ]:
# === Visualisation AFC ===

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# --- 1. Individus — couleur cluster baseline ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[0].scatter(F_afc[mask, 0], F_afc[mask, 1],
                    s=12, alpha=0.5, color=cluster_colors[c], label=f'C{c+1}')
axes[0].set_xlabel(f'Axe 1 ({afc_ax1:.1f}%)', fontweight='bold')
axes[0].set_ylabel(f'Axe 2 ({afc_ax2:.1f}%)', fontweight='bold')
axes[0].set_title('AFC — Individus\n(couleur = cluster K-means)', fontweight='bold')
axes[0].legend(title='Cluster', fontsize=8)
axes[0].axhline(0, color='gray', lw=0.5, ls='--')
axes[0].axvline(0, color='gray', lw=0.5, ls='--')

# --- 2. Variables (groupes planctoniques) ---
colors_sp = plt.cm.tab20(np.linspace(0, 1, len(COLS_CONC)))
for i, (col, xy) in enumerate(zip(COLS_CONC, G_afc)):
    x, y = xy
    axes[1].annotate('', xy=(x, y), xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color=colors_sp[i], lw=1.2))
    axes[1].text(x * 1.08, y * 1.08, col, fontsize=7.5, color=colors_sp[i],
                 fontweight='bold', ha='center', va='center')
axes[1].axhline(0, color='gray', lw=0.5, ls='--')
axes[1].axvline(0, color='gray', lw=0.5, ls='--')
axes[1].set_xlabel(f'Axe 1 ({afc_ax1:.1f}%)', fontweight='bold')
axes[1].set_ylabel(f'Axe 2 ({afc_ax2:.1f}%)', fontweight='bold')
axes[1].set_title('AFC — Variables (groupes planctoniques)', fontweight='bold')

# --- 3. Biplot combiné (individus + variables) ---
q_ind = np.percentile(np.abs(F_afc), 85) + 1e-12
q_var = np.percentile(np.abs(G_afc), 85) + 1e-12
scale = q_ind / q_var * 0.6

axes[2].scatter(F_afc[:, 0], F_afc[:, 1], s=6, alpha=0.15,
                color='lightsteelblue', label='Individus')
for i, (col, xy) in enumerate(zip(COLS_CONC, G_afc)):
    xs, ys = xy[0] * scale, xy[1] * scale
    axes[2].annotate('', xy=(xs, ys), xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='crimson', lw=1.0, alpha=0.8))
    axes[2].text(xs * 1.1, ys * 1.1, col, fontsize=6.5, color='darkred',
                 ha='center', va='center')
axes[2].axhline(0, color='gray', lw=0.5, ls='--')
axes[2].axvline(0, color='gray', lw=0.5, ls='--')
axes[2].set_xlabel(f'Axe 1 ({afc_ax1:.1f}%)', fontweight='bold')
axes[2].set_ylabel(f'Axe 2 ({afc_ax2:.1f}%)', fontweight='bold')
axes[2].set_title('AFC — Biplot (individus + variables)', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle(f'Analyse Factorielle des Correspondances (AFC) — Inertie axes 1+2 : {afc_cum2:.1f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_14_afc.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.2 Isomap (Tenenbaum et al., 2000)

L'Isomap est une méthode **non-linéaire** qui généralise l'MDS en utilisant les **distances géodésiques** sur la variété des données (plutôt que les distances euclidiennes directes).

**Algorithme en 3 étapes** :
1. Construire le graphe des $k$ plus proches voisins
2. Calculer les distances géodésiques (chemin le plus court dans le graphe)
3. Appliquer l'MDS classique sur la matrice de distances géodésiques

**Paramètre clé** : `n_neighbors` : contrôle le compromis entre fidélité locale (petit $k$) et globale (grand $k$).  
**Mesure de qualité** : Erreur de reconstruction (plus faible = meilleure représentation).

In [ ]:
# === Isomap — Optimisation du nombre de voisins ===

n_neighbors_range = [5, 10, 15, 20, 30]
isomap_results = {}

print('Optimisation Isomap (n_neighbors) :')
print(f'{"n_neighbors":>12} | {"Err. reconstruction":>20} | {"Temps (s)":>10}')
print('-' * 48)

for n_nbrs in n_neighbors_range:
    t0 = time.time()
    try:
        iso = Isomap(n_neighbors=n_nbrs, n_components=2, n_jobs=-1)
        X_iso_tmp = iso.fit_transform(X_hell)
        err = iso.reconstruction_error()
        t_iso = time.time() - t0
        isomap_results[n_nbrs] = {'embedding': X_iso_tmp, 'error': err, 'time': t_iso}
        print(f'{n_nbrs:>12} | {err:>20.4f} | {t_iso:>10.2f}')
    except Exception as e:
        print(f'{n_nbrs:>12} | Erreur : {str(e)[:35]}')

# Meilleur n_neighbors : minimise l'erreur de reconstruction
best_n_iso = min(isomap_results, key=lambda k: isomap_results[k]['error'])
X_isomap   = isomap_results[best_n_iso]['embedding']
t_isomap   = isomap_results[best_n_iso]['time']
iso_err    = isomap_results[best_n_iso]['error']

print(f'')
print(f'Meilleur n_neighbors = {best_n_iso} (err. reconstruction = {iso_err:.4f})')

In [ ]:
# === Isomap — Visualisation ===

# Palette couleurs par couche (réutilisée pour LLE et MDS)
unique_layers = sorted(df['layer'].dropna().unique())
cmap_layers   = plt.cm.Set1(np.linspace(0, 0.8, len(unique_layers)))

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Courbe erreur de reconstruction
nbrs_list = sorted(isomap_results.keys())
errors_iso = [isomap_results[n]['error'] for n in nbrs_list]
axes[0].plot(nbrs_list, errors_iso, 'o-', color='steelblue', markersize=8, linewidth=2)
axes[0].axvline(best_n_iso, color='red', ls='--', alpha=0.8, label=f'Optimal k={best_n_iso}')
axes[0].set_xlabel('n_neighbors', fontweight='bold')
axes[0].set_ylabel('Erreur de reconstruction', fontweight='bold')
axes[0].set_title('Isomap — Choix du nombre de voisins', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Projection 2D — couleur cluster
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_isomap[mask, 0], X_isomap[mask, 1],
                    s=12, alpha=0.5, color=cluster_colors[c], label=f'C{c+1}')
axes[1].set_xlabel('Isomap dim 1', fontweight='bold')
axes[1].set_ylabel('Isomap dim 2', fontweight='bold')
axes[1].set_title(f'Isomap (k={best_n_iso}) — Couleur cluster', fontweight='bold')
axes[1].legend(title='Cluster', fontsize=8)
axes[1].axhline(0, color='gray', lw=0.5, ls='--')
axes[1].axvline(0, color='gray', lw=0.5, ls='--')

# Projection 2D — couleur couche
for k, layer in enumerate(unique_layers):
    mask = df['layer'].values == layer
    axes[2].scatter(X_isomap[mask, 0], X_isomap[mask, 1],
                    s=12, alpha=0.5, color=cmap_layers[k], label=layer)
axes[2].set_xlabel('Isomap dim 1', fontweight='bold')
axes[2].set_ylabel('Isomap dim 2', fontweight='bold')
axes[2].set_title('Isomap — Couleur couche océanique', fontweight='bold')
axes[2].legend(title='Couche', fontsize=8)

plt.suptitle(f'Isomap (n_neighbors={best_n_iso}) — Erreur reconstruction = {iso_err:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_15_isomap.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.3 LLE : Locally Linear Embedding (Roweis & Saul, 2000)

Le LLE suppose que chaque point peut être **reconstruit linéairement** à partir de ses $k$ voisins les plus proches, et cherche à préserver ces poids de reconstruction dans l'espace réduit.

**Algorithme en 3 étapes** :
1. Trouver les $k$ plus proches voisins de chaque point
2. Calculer les poids de reconstruction locale (moindres carrés sous contrainte)
3. Chercher l'embedding de basse dimension qui minimise l'erreur de reconstruction globale

**Avantage** : Capture des géométries **locales** non-linéaires  
**Limite** : Sensible au bruit, instable pour les petits `n_neighbors`, sensible aux outliers

In [ ]:
# === LLE — Optimisation du nombre de voisins ===

n_neighbors_lle = [5, 10, 15, 20]
lle_results = {}
t_lle_final = 0.0   # fallback
lle_err_final = 0.0
best_n_lle = 5

print('Optimisation LLE (n_neighbors, method=standard) :')
print(f'{"n_neighbors":>12} | {"Err. reconstruction":>20} | {"Temps (s)":>10}')
print('-' * 48)

for n_nbrs in n_neighbors_lle:
    t0 = time.time()
    try:
        lle = LocallyLinearEmbedding(
            n_neighbors=n_nbrs, n_components=2,
            method='standard', random_state=RANDOM_STATE, n_jobs=-1
        )
        X_lle_tmp = lle.fit_transform(X_hell)
        err = lle.reconstruction_error_
        t_lle = time.time() - t0
        lle_results[n_nbrs] = {'embedding': X_lle_tmp, 'error': err, 'time': t_lle}
        print(f'{n_nbrs:>12} | {err:>20.6f} | {t_lle:>10.2f}')
    except Exception as e:
        print(f'{n_nbrs:>12} | Erreur : {str(e)[:40]}')

if lle_results:
    best_n_lle    = min(lle_results, key=lambda k: lle_results[k]['error'])
    X_lle         = lle_results[best_n_lle]['embedding']
    t_lle_final   = lle_results[best_n_lle]['time']
    lle_err_final = lle_results[best_n_lle]['error']
    print(f'')
    print(f'Meilleur LLE : n_neighbors={best_n_lle} (err={lle_err_final:.6f})')
else:
    print('LLE indisponible — fallback sur ACP 2D.')
    X_lle = X_pca[:, :2].copy()

In [ ]:
# === LLE — Visualisation ===

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Courbe erreur de reconstruction
if lle_results:
    nbrs_lle_list = sorted(lle_results.keys())
    errors_lle = [lle_results[n]['error'] for n in nbrs_lle_list]
    axes[0].plot(nbrs_lle_list, errors_lle, 's-', color='mediumseagreen', markersize=8, linewidth=2)
    axes[0].axvline(best_n_lle, color='red', ls='--', alpha=0.8, label=f'Optimal k={best_n_lle}')
    axes[0].legend()
axes[0].set_xlabel('n_neighbors', fontweight='bold')
axes[0].set_ylabel('Erreur de reconstruction', fontweight='bold')
axes[0].set_title('LLE — Sensibilité au nombre de voisins', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Projection 2D — couleur cluster
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_lle[mask, 0], X_lle[mask, 1],
                    s=12, alpha=0.5, color=cluster_colors[c], label=f'C{c+1}')
axes[1].set_xlabel('LLE dim 1', fontweight='bold')
axes[1].set_ylabel('LLE dim 2', fontweight='bold')
axes[1].set_title(f'LLE (k={best_n_lle}) — Couleur cluster', fontweight='bold')
axes[1].legend(title='Cluster', fontsize=8)
axes[1].axhline(0, color='gray', lw=0.5, ls='--')
axes[1].axvline(0, color='gray', lw=0.5, ls='--')

# Projection 2D — couleur couche
for k, layer in enumerate(unique_layers):
    mask = df['layer'].values == layer
    axes[2].scatter(X_lle[mask, 0], X_lle[mask, 1],
                    s=12, alpha=0.5, color=cmap_layers[k], label=layer)
axes[2].set_xlabel('LLE dim 1', fontweight='bold')
axes[2].set_ylabel('LLE dim 2', fontweight='bold')
axes[2].set_title('LLE — Couleur couche océanique', fontweight='bold')
axes[2].legend(title='Couche', fontsize=8)

plt.suptitle(f'LLE (n_neighbors={best_n_lle}) — Erreur reconstruction = {lle_err_final:.6f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_16_lle.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.4 MDS : Multidimensional Scaling (Kruskal, 1964)

Le MDS représente les données dans un espace de basse dimension en **préservant les distances** entre individus. Deux variantes sont comparées :

| Variante | Description | Critère d'optimisation |
|----------|-------------|------------------------|
| **MDS métrique** | Préserve les distances euclidiennes | Minimise le STRESS (différences de distances) |
| **MDS non-métrique** | Préserve l'ordre des distances | Minimise le STRESS monotone |

**Mesure de qualité : Stress de Kruskal** :

$$\text{stress} = \sqrt{\frac{\sum (d_{ij} - \hat{d}_{ij})^2}{\sum d_{ij}^2}}$$

Interprétation : < 0.05 excellent | < 0.10 bon | < 0.20 acceptable | > 0.20 médiocre

In [ ]:
# === MDS Métrique et Non-métrique ===

# Sous-échantillonnage (MDS en O(n²) mémoire)
N_MDS = min(2000, len(X_hell))
rng_mds = np.random.RandomState(RANDOM_STATE)
idx_mds = rng_mds.choice(len(X_hell), N_MDS, replace=False)
X_mds_input = X_hell[idx_mds]
labels_mds  = labels_final[idx_mds]
df_mds      = df.iloc[idx_mds].reset_index(drop=True)

print(f'Sous-échantillon pour MDS : {N_MDS} observations')

# MDS Métrique
t0 = time.time()
try:
    mds_m = MDS(n_components=2, metric=True, random_state=RANDOM_STATE, n_jobs=-1)
    X_mds_metric = mds_m.fit_transform(X_mds_input)
    stress_metric = mds_m.stress_
except TypeError:
    mds_m = MDS(n_components=2, metric=True, random_state=RANDOM_STATE)
    X_mds_metric = mds_m.fit_transform(X_mds_input)
    stress_metric = mds_m.stress_
t_mds_m = time.time() - t0

# MDS Non-métrique
t0 = time.time()
try:
    mds_nm = MDS(n_components=2, metric=False, random_state=RANDOM_STATE, n_jobs=-1, max_iter=300)
    X_mds_nonmetric = mds_nm.fit_transform(X_mds_input)
    stress_nonmetric = mds_nm.stress_
except TypeError:
    mds_nm = MDS(n_components=2, metric=False, random_state=RANDOM_STATE, max_iter=300)
    X_mds_nonmetric = mds_nm.fit_transform(X_mds_input)
    stress_nonmetric = mds_nm.stress_
t_mds_nm = time.time() - t0

def stress_qual(s):
    if s < 0.05: return 'Excellent'
    if s < 0.10: return 'Bon'
    if s < 0.20: return 'Acceptable'
    return 'Médiocre'

print('')
print('=== MDS — Résultats ===')
print(f'MDS Métrique     : stress = {stress_metric:.6f} ({stress_qual(stress_metric)}) | temps = {t_mds_m:.1f}s')
print(f'MDS Non-métrique : stress = {stress_nonmetric:.6f} ({stress_qual(stress_nonmetric)}) | temps = {t_mds_nm:.1f}s')

In [ ]:
# === MDS — Visualisation ===

from sklearn.metrics import pairwise_distances as pw_dist

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# MDS Métrique — couleur cluster
for c in range(K_OPTIMAL):
    mask = labels_mds == c
    axes[0, 0].scatter(X_mds_metric[mask, 0], X_mds_metric[mask, 1],
                       s=15, alpha=0.5, color=cluster_colors[c], label=f'C{c+1}')
axes[0, 0].set_xlabel('Dim 1', fontweight='bold')
axes[0, 0].set_ylabel('Dim 2', fontweight='bold')
axes[0, 0].set_title(f'MDS Métrique (stress={stress_metric:.4f})', fontweight='bold')
axes[0, 0].legend(title='Cluster', fontsize=8)
axes[0, 0].axhline(0, color='gray', lw=0.5, ls='--')
axes[0, 0].axvline(0, color='gray', lw=0.5, ls='--')

# MDS Non-métrique — couleur cluster
for c in range(K_OPTIMAL):
    mask = labels_mds == c
    axes[0, 1].scatter(X_mds_nonmetric[mask, 0], X_mds_nonmetric[mask, 1],
                       s=15, alpha=0.5, color=cluster_colors[c], label=f'C{c+1}')
axes[0, 1].set_xlabel('Dim 1', fontweight='bold')
axes[0, 1].set_ylabel('Dim 2', fontweight='bold')
axes[0, 1].set_title(f'MDS Non-métrique (stress={stress_nonmetric:.4f})', fontweight='bold')
axes[0, 1].legend(title='Cluster', fontsize=8)
axes[0, 1].axhline(0, color='gray', lw=0.5, ls='--')
axes[0, 1].axvline(0, color='gray', lw=0.5, ls='--')

# MDS Non-métrique — couleur couche
layers_mds_vals = df_mds['layer'].values
for k, layer in enumerate(unique_layers):
    mask = layers_mds_vals == layer
    if mask.sum() > 0:
        axes[1, 0].scatter(X_mds_nonmetric[mask, 0], X_mds_nonmetric[mask, 1],
                           s=15, alpha=0.5, color=cmap_layers[k], label=layer)
axes[1, 0].set_xlabel('Dim 1', fontweight='bold')
axes[1, 0].set_ylabel('Dim 2', fontweight='bold')
axes[1, 0].set_title('MDS Non-métrique — Couche océanique', fontweight='bold')
axes[1, 0].legend(title='Couche', fontsize=8)

# Diagramme de Shepard
n_shep = min(400, N_MDS)
D_orig = pw_dist(X_mds_input[:n_shep])
D_emb  = pw_dist(X_mds_nonmetric[:n_shep])
tril   = np.tril_indices(n_shep, -1)
axes[1, 1].scatter(D_orig[tril], D_emb[tril], s=1, alpha=0.08, color='steelblue')
axes[1, 1].set_xlabel('Distances originales (espace Hellinger)', fontweight='bold')
axes[1, 1].set_ylabel("Distances dans l'embedding MDS", fontweight='bold')
axes[1, 1].set_title(f'Diagramme de Shepard (MDS non-métrique, n={n_shep})', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('MDS Métrique et Non-métrique', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_17_mds.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.5 Comparaison des Représentations

Cette section évalue et compare les cinq représentations selon :
- **Qualité géométrique** : variance/inertie expliquée, stress, erreur de reconstruction
- **Séparabilité des clusters** : score de silhouette dans l'espace 2D réduit
- **Temps de calcul**
- **Interprétabilité biologique** des axes

In [ ]:
# === Scores de silhouette dans chaque espace 2D ===

n_sil = min(3000, len(labels_final))
idx_sil = np.random.RandomState(RANDOM_STATE).choice(len(labels_final), n_sil, replace=False)

sil_pca    = sil_score(X_pca[idx_sil, :2], labels_final[idx_sil],
                        sample_size=1000, random_state=RANDOM_STATE)
sil_afc    = sil_score(F_afc[idx_sil, :2], labels_final[idx_sil],
                        sample_size=1000, random_state=RANDOM_STATE)
sil_isomap = sil_score(X_isomap[idx_sil, :2], labels_final[idx_sil],
                        sample_size=1000, random_state=RANDOM_STATE)
sil_lle    = sil_score(X_lle[idx_sil, :2], labels_final[idx_sil],
                        sample_size=1000, random_state=RANDOM_STATE)

# MDS (sous-échantillon spécifique)
mds_idx_set = set(idx_mds.tolist())
common_idx  = [i for i in idx_sil if i in mds_idx_set]
if len(common_idx) > 200:
    mds_pos = {v: i for i, v in enumerate(idx_mds)}
    sub_pos = [mds_pos[i] for i in common_idx]
    sub_lbl = [labels_mds[mds_pos[i]] for i in common_idx]
    sil_mds = sil_score(X_mds_nonmetric[sub_pos], sub_lbl,
                         sample_size=min(500, len(sub_pos)), random_state=RANDOM_STATE)
else:
    sil_mds = None

print('=== Scores de Silhouette (espace 2D) ===')
print(f'ACP               : {sil_pca:.4f}')
print(f'AFC               : {sil_afc:.4f}')
print(f'Isomap (k={best_n_iso})    : {sil_isomap:.4f}')
print(f'LLE (k={best_n_lle})       : {sil_lle:.4f}')
if sil_mds:
    print(f'MDS Non-métrique  : {sil_mds:.4f}')

print('')
print('=== Tableau comparatif des représentations ===')
print(f'{"Méthode":<22} | {"Type":<12} | {"Qualité":<24} | {"Silhouette":>10} | {"Temps":>8}')
print('-' * 85)

rows_comp = [
    ('ACP',              'Linéaire',     f'Var. 2CP = {pc1_var+pc2_var:.1f}%',   sil_pca,    '<1s'),
    ('AFC',              'Linéaire',     f'Inertie = {afc_cum2:.1f}%',            sil_afc,    f'{t_afc:.1f}s'),
    (f'Isomap (k={best_n_iso})',  'Non-linéaire', f'Err. = {iso_err:.4f}',        sil_isomap, f'{t_isomap:.1f}s'),
    (f'LLE (k={best_n_lle})',     'Non-linéaire', f'Err. = {lle_err_final:.5f}',  sil_lle,    f'{t_lle_final:.1f}s'),
    ('MDS Non-métrique', 'Non-linéaire', f'Stress = {stress_nonmetric:.4f}',      sil_mds,    f'{t_mds_nm:.1f}s'),
]
for meth, mtype, qual, sil, tstr in rows_comp:
    s = f'{sil:.4f}' if sil is not None else 'N/A'
    print(f'{meth:<22} | {mtype:<12} | {qual:<24} | {s:>10} | {tstr:>8}')

In [ ]:
# === Comparaison visuelle — 5 représentations côte à côte ===

fig, axes = plt.subplots(2, 3, figsize=(22, 13))
ax_list = axes.flatten()

embeddings = [
    (X_pca[:, :2],    labels_final, f'ACP — PC1+PC2 ({pc1_var+pc2_var:.1f}%)',        'PC1',   'PC2'),
    (F_afc[:, :2],    labels_final, f'AFC — Axes 1+2 ({afc_cum2:.1f}%)',               'Axe 1', 'Axe 2'),
    (X_isomap,        labels_final, f'Isomap (k={best_n_iso}, err={iso_err:.3f})',     'Dim 1', 'Dim 2'),
    (X_lle,           labels_final, f'LLE (k={best_n_lle}, err={lle_err_final:.4f})', 'Dim 1', 'Dim 2'),
    (X_mds_nonmetric, labels_mds,   f'MDS Non-métrique (stress={stress_nonmetric:.4f})', 'Dim 1', 'Dim 2'),
]

for i, (X_emb, lbls, title, xl, yl) in enumerate(embeddings):
    for c in range(K_OPTIMAL):
        mask = lbls == c
        ax_list[i].scatter(X_emb[mask, 0], X_emb[mask, 1],
                            s=10, alpha=0.4, color=cluster_colors[c], label=f'C{c+1}')
    ax_list[i].set_xlabel(xl, fontweight='bold', fontsize=10)
    ax_list[i].set_ylabel(yl, fontweight='bold', fontsize=10)
    ax_list[i].set_title(title, fontweight='bold', fontsize=10)
    ax_list[i].legend(title='Cluster', fontsize=7, markerscale=1.5)
    ax_list[i].axhline(0, color='gray', lw=0.4, ls='--')
    ax_list[i].axvline(0, color='gray', lw=0.4, ls='--')

ax_list[5].set_visible(False)

plt.suptitle('Comparaison des 5 méthodes de réduction dimensionnelle\n(couleur = cluster K-means baseline)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_18_comparaison_representations.png', dpi=120, bbox_inches='tight')
plt.show()

# Meilleure représentation
valid_sils = [('ACP', sil_pca), ('AFC', sil_afc),
              ('Isomap', sil_isomap), ('LLE', sil_lle)]
if sil_mds is not None:
    valid_sils.append(('MDS Non-métrique', sil_mds))
best_repr_name, best_sil_val = max(valid_sils, key=lambda x: x[1])

print(f'Meilleure représentation (silhouette 2D) : {best_repr_name} ({best_sil_val:.4f})')
print(f'=> Jalon 3 : clustering sur {n_80} composantes ACP (cohérence baseline)')

# Dictionnaire de référence pour Jalon 3
representations = {
    'ACP':    X_pca[:, :2],
    'AFC':    F_afc[:, :2],
    'Isomap': X_isomap,
    'LLE':    X_lle,
}
print('Dictionnaire "representations" disponible pour le Jalon 3.')

## 3.6 Synthèse du Jalon 2

### Interprétation comparative

| Méthode | Points forts | Limites | Usage recommandé |
|---------|-------------|---------|-----------------|
| **ACP** | Variance maximale, axes interprétables | Relations linéaires seulement | Référence, interprétation |
| **AFC** | Adapté aux profils compositionnels, biplot joint | Sensible aux espèces rares | Associations taxons/sites |
| **Isomap** | Géométrie globale non-linéaire | Sensible aux outliers, lent | Structure topologique |
| **LLE** | Géométrie locale fine | Instable, sensible au bruit | Voisinages locaux |
| **MDS non-métrique** | Robuste, préserve les rangs | Stress potentiellement élevé | Comparaison par distances |

### Conclusions biologiques

Les méthodes non-linéaires (Isomap, LLE) révèlent une **structure plus complexe** que l'ACP, cohérente avec :
- Un **gradient bathymétrique** fort (épipélagique ↔ méso/bathypélagique)
- Une **saisonnalité** dans la composition des communautés
- Des **associations préférentielles** entre certains groupes taxonomiques

L'AFC met en évidence les associations entre échantillons et taxons via le biplot, offrant une interprétation directe en termes d'écologie des communautés.

### Transition vers le Jalon 3

Le clustering sera appliqué sur les **`n_80` composantes ACP** (cohérence avec la baseline), en testant des algorithmes capables de détecter des formes arbitraires : DBSCAN, HDBSCAN, OPTICS, Spectral Clustering, et GMM.

---

## Références bibliographiques : Jalon 2

- **Tenenbaum, J.B., de Silva, V. & Langford, J.C. (2000)** : A global geometric framework for nonlinear dimensionality reduction. *Science*, 290(5500), 2319–2323.
- **Roweis, S.T. & Saul, L.K. (2000)** : Nonlinear dimensionality reduction by locally linear embedding. *Science*, 290(5500), 2323–2326.
- **Kruskal, J.B. (1964)** : Multidimensional scaling by optimizing goodness of fit to a nonmetric hypothesis. *Psychometrika*, 29(1), 1–27.
- **Benzécri, J.P. (1973)** : *L'Analyse des données. Vol. 2 : L'analyse des correspondances*. Dunod, Paris.

---

# Partie 4 : Nouveaux Algorithmes de Clustering (Jalon 3)

## Objectif

Explorer des algorithmes de clustering alternatifs au K-means, chacun avec une philosophie différente de la structure des données :

| Algorithme | Type | Approche | Paramètre clé |
|---|---|---|---|
| **DBSCAN** | Densité | Régions denses + bruit | ε, min_samples |
| **HDBSCAN** | Densité hiérarchique | Stabilité des clusters | min_cluster_size |
| **OPTICS** | Densité ordonnée | Multi-densité, reachability | min_samples |
| **Spectral** | Graphe spectral | Connexions globales | n_clusters |
| **GMM** | Probabiliste | Mélanges gaussiens | n_components, covariance |

Tous les algorithmes sont appliqués sur le **même espace PCA à `n_80` composantes** que la baseline K-means, permettant une comparaison équitable des méthodes de clustering.

In [ ]:
# === Imports supplémentaires — Jalon 3 ===
from sklearn.cluster import DBSCAN, OPTICS, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             v_measure_score)

# Détection de HDBSCAN (package séparé ou sklearn ≥ 1.3)
try:
    import hdbscan as hdbscan_lib
    HDBSCAN_LIB = True
    print("hdbscan (package) disponible")
except ImportError:
    try:
        from sklearn.cluster import HDBSCAN as hdbscan_lib
        HDBSCAN_LIB = False
        print("sklearn.cluster.HDBSCAN disponible")
    except ImportError:
        hdbscan_lib = None
        print("HDBSCAN non disponible")

import warnings
warnings.filterwarnings('ignore')

# Espace de clustering identique à la baseline
X_clust = X_pca[:, :n_80]
print(f"\nEspace de clustering : {X_clust.shape}  ({n_80} composantes PCA, ≥80% variance)")
print(f"Baseline K-means    : k={best_sil_k} clusters")

# Dictionnaire centralisé pour tous les clusterings
all_clusterings = {'K-means': labels_final}
metrics_all     = {}

def compute_metrics(X, labels, ref_labels=None):
    """Métriques internes + (optionnel) externes vs ref_labels."""
    mask         = labels != -1
    labels_clean = labels[mask]
    X_clean      = X[mask]
    n_clusters   = len(np.unique(labels_clean))
    n_noise      = int((labels == -1).sum())

    if n_clusters < 2:
        return dict(n_clusters=n_clusters, n_noise=n_noise,
                    silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan,
                    ari=np.nan, nmi=np.nan, v_measure=np.nan)

    n_sil = min(5000, len(X_clean))
    sil   = silhouette_score(X_clean, labels_clean, sample_size=n_sil, random_state=42)
    db    = davies_bouldin_score(X_clean, labels_clean)
    ch    = calinski_harabasz_score(X_clean, labels_clean)

    ari = nmi = vm = np.nan
    if ref_labels is not None:
        ref_clean = ref_labels[mask]
        ari = adjusted_rand_score(ref_clean, labels_clean)
        nmi = normalized_mutual_info_score(ref_clean, labels_clean)
        vm  = v_measure_score(ref_clean, labels_clean)

    return dict(n_clusters=n_clusters, n_noise=n_noise,
                silhouette=sil, davies_bouldin=db, calinski_harabasz=ch,
                ari=ari, nmi=nmi, v_measure=vm)

# Métriques de la baseline K-means (pas de bruit, ref = elle-même)
metrics_all['K-means'] = compute_metrics(X_clust, labels_final)
metrics_all['K-means'].update(ari=1.0, nmi=1.0, v_measure=1.0)  # identique à elle-même

print(f"\nBaseline K-means :")
print(f"  Silhouette     : {metrics_all['K-means']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['K-means']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['K-means']['calinski_harabasz']:.1f}")

## 4.1 DBSCAN : Density-Based Spatial Clustering of Applications with Noise (Ester et al., 1996)

DBSCAN classe les points selon leur densité locale. Il identifie trois types de points :
- **Points cœurs** : au moins `min_samples` voisins dans un rayon ε
- **Points frontières** : dans le voisinage d'un cœur, mais pas cœurs eux-mêmes
- **Points de bruit** : isolés, sans appartenance à un cluster (label = -1)

**Avantages :** pas de k à fixer, détecte le bruit, formes arbitraires  
**Limites :** sensible à ε, difficultés avec des densités variables

Le paramètre ε est estimé via le **graphe des distances au k-ème voisin** : le "coude" indique le seuil de séparation densité/bruit.

In [ ]:
# === DBSCAN ===

N_DBSCAN = min(5000, len(X_clust))
rng_db   = np.random.default_rng(42)
idx_db   = rng_db.choice(len(X_clust), N_DBSCAN, replace=False)
X_db     = X_clust[idx_db]

# --- 1. Graphe k-NN pour estimer epsilon ---
k_nn              = 5
nbrs              = NearestNeighbors(n_neighbors=k_nn).fit(X_db)
knn_distances, _  = nbrs.kneighbors(X_db)
knn_dists_sorted  = np.sort(knn_distances[:, -1])

# Candidats epsilon = percentiles de la distribution des distances k-NN
eps_candidates    = np.percentile(knn_dists_sorted, [70, 80, 90, 95])
min_samples_vals  = [3, 5, 10, 15]

# --- 2. Grid search ---
db_results = []
for eps in eps_candidates:
    for ms in min_samples_vals:
        db   = DBSCAN(eps=eps, min_samples=ms).fit(X_db)
        labs = db.labels_
        n_cl = len(set(labs)) - (1 if -1 in labs else 0)
        noise_p = (labs == -1).sum() / len(labs) * 100
        if n_cl >= 2 and (labs != -1).sum() > n_cl:
            sil = silhouette_score(X_db[labs != -1], labs[labs != -1],
                                   sample_size=min(1000, (labs != -1).sum()), random_state=42)
        else:
            sil = np.nan
        db_results.append({'eps': round(eps, 3), 'min_samples': ms,
                           'n_clusters': n_cl, 'noise_pct': round(noise_p, 1),
                           'silhouette': round(sil, 4) if not np.isnan(sil) else np.nan})

df_dbscan_grid = pd.DataFrame(db_results)
print("Grid search DBSCAN :")
print(df_dbscan_grid.to_string(index=False))

# Meilleurs paramètres
valid_db = df_dbscan_grid[(df_dbscan_grid['n_clusters'] >= 2) &
                           (df_dbscan_grid['noise_pct'] < 40) &
                           (df_dbscan_grid['silhouette'].notna())]
if len(valid_db):
    best_db  = valid_db.loc[valid_db['silhouette'].idxmax()]
    BEST_EPS = best_db['eps']
    BEST_MS  = int(best_db['min_samples'])
else:
    BEST_EPS, BEST_MS = eps_candidates[1], 5

print(f"\nMeilleurs paramètres : eps={BEST_EPS}, min_samples={BEST_MS}")

# Application sur données complètes
dbscan_full     = DBSCAN(eps=BEST_EPS, min_samples=BEST_MS).fit(X_clust)
labels_dbscan   = dbscan_full.labels_
n_cl_db         = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
noise_db        = (labels_dbscan == -1).sum()
print(f"  → {n_cl_db} clusters, {noise_db} bruit ({noise_db/len(labels_dbscan)*100:.1f}%)")

metrics_all['DBSCAN']      = compute_metrics(X_clust, labels_dbscan, labels_final)
all_clusterings['DBSCAN']  = labels_dbscan

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. k-NN distance plot
axes[0].plot(knn_dists_sorted, color='steelblue', lw=1.5)
axes[0].axhline(BEST_EPS, color='crimson', linestyle='--', label=f'ε={BEST_EPS:.3f}')
axes[0].set_xlabel("Points triés", fontsize=12)
axes[0].set_ylabel(f"Distance au {k_nn}e voisin", fontsize=12)
axes[0].set_title("Graphe k-NN (choix de ε)", fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

# b. Clusters PC1-PC2
for lbl in sorted(set(labels_dbscan)):
    mask = labels_dbscan == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"DBSCAN — {n_cl_db} clusters (ε={BEST_EPS:.3f}, ms={BEST_MS})", fontsize=13)
if n_cl_db <= 10:
    axes[1].legend(fontsize=9, markerscale=2)
axes[1].grid(alpha=0.2)

# c. Profil biologique par cluster
if n_cl_db >= 2:
    df_db_bio = df_hellinger.copy()
    df_db_bio['cluster'] = labels_dbscan
    df_db_bio = df_db_bio[df_db_bio['cluster'] != -1]
    bio_db = df_db_bio.groupby('cluster')[COLS_CONC].mean()
    im = axes[2].imshow(bio_db.values.T, aspect='auto', cmap='YlOrRd')
    axes[2].set_xticks(range(len(bio_db)))
    axes[2].set_xticklabels([f'C{i+1}' for i in bio_db.index], fontsize=10)
    axes[2].set_yticks(range(len(COLS_CONC)))
    axes[2].set_yticklabels(COLS_CONC, fontsize=8)
    plt.colorbar(im, ax=axes[2], label='Hellinger (moy.)')
    axes[2].set_title("Profil biologique DBSCAN", fontsize=13)

plt.tight_layout()
plt.savefig('fig_19_dbscan.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques DBSCAN :")
print(f"  Silhouette     : {metrics_all['DBSCAN']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['DBSCAN']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['DBSCAN']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['DBSCAN']['ari']:.4f}")
print(f"  Points de bruit: {noise_db} ({noise_db/len(labels_dbscan)*100:.1f}%)")

## 4.2 HDBSCAN : Hierarchical DBSCAN (Campello et al., 2013)

HDBSCAN étend DBSCAN en construisant une **hiérarchie de clusters** basée sur la densité, puis en extrayant les clusters les plus stables selon un critère de persistance. Il est plus robuste que DBSCAN face aux variations de densité et ne nécessite pas de fixer ε.

**Paramètres clés :**
- `min_cluster_size` : taille minimale d'un cluster stable
- `min_samples` : robustesse au bruit (optionnel, par défaut = `min_cluster_size`)

**Avantages :** gère les densités variables, hiérarchie extraite automatiquement, robuste  
**Limites :** plus lent sur grands jeux de données, sensible à `min_cluster_size`

In [ ]:
# === HDBSCAN ===

mcs_vals     = [10, 20, 50, 100, 200]
ms_vals_hdb  = [None, 5, 10]
hdb_results  = []

N_HDB    = min(5000, len(X_clust))
rng_hdb  = np.random.default_rng(42)
idx_hdb  = rng_hdb.choice(len(X_clust), N_HDB, replace=False)
X_hdb    = X_clust[idx_hdb]

if hdbscan_lib is not None:
    print("Grid search HDBSCAN (min_cluster_size × min_samples)...")
    for mcs in mcs_vals:
        for ms in ms_vals_hdb:
            try:
                if HDBSCAN_LIB:
                    params = {'min_cluster_size': mcs}
                    if ms is not None:
                        params['min_samples'] = ms
                    clusterer = hdbscan_lib.HDBSCAN(**params)
                    clusterer.fit(X_hdb)
                    labs = clusterer.labels_
                else:
                    params = {'min_cluster_size': mcs}
                    if ms is not None:
                        params['min_samples'] = ms
                    clusterer = hdbscan_lib(**params)
                    labs = clusterer.fit_predict(X_hdb)

                n_cl     = len(set(labs)) - (1 if -1 in labs else 0)
                noise_p  = (labs == -1).sum() / len(labs) * 100
                if n_cl >= 2 and (labs != -1).sum() > n_cl:
                    sil = silhouette_score(X_hdb[labs != -1], labs[labs != -1],
                                          sample_size=min(1000, (labs != -1).sum()), random_state=42)
                else:
                    sil = np.nan
                hdb_results.append({'min_cluster_size': mcs, 'min_samples': ms or 'auto',
                                    'n_clusters': n_cl, 'noise_pct': round(noise_p, 1),
                                    'silhouette': round(sil, 4) if not np.isnan(sil) else np.nan})
            except Exception:
                hdb_results.append({'min_cluster_size': mcs, 'min_samples': ms or 'auto',
                                    'n_clusters': -1, 'noise_pct': np.nan, 'silhouette': np.nan})

    df_hdb_grid = pd.DataFrame(hdb_results)
    print(df_hdb_grid.to_string(index=False))

    valid_hdb = df_hdb_grid[(df_hdb_grid['n_clusters'] >= 2) &
                             (df_hdb_grid['noise_pct'] < 50) &
                             (df_hdb_grid['silhouette'].notna())]
    if len(valid_hdb):
        best_h      = valid_hdb.loc[valid_hdb['silhouette'].idxmax()]
        BEST_MCS    = int(best_h['min_cluster_size'])
        BEST_MS_HDB = None if best_h['min_samples'] == 'auto' else int(best_h['min_samples'])
    else:
        BEST_MCS, BEST_MS_HDB = 50, None

    print(f"\nMeilleurs paramètres : min_cluster_size={BEST_MCS}, min_samples={BEST_MS_HDB}")

    # Application sur données complètes
    try:
        if HDBSCAN_LIB:
            pf = {'min_cluster_size': BEST_MCS}
            if BEST_MS_HDB:
                pf['min_samples'] = BEST_MS_HDB
            hdb_clf = hdbscan_lib.HDBSCAN(**pf)
            hdb_clf.fit(X_clust)
            labels_hdbscan = hdb_clf.labels_
        else:
            pf = {'min_cluster_size': BEST_MCS}
            if BEST_MS_HDB:
                pf['min_samples'] = BEST_MS_HDB
            labels_hdbscan = hdbscan_lib(**pf).fit_predict(X_clust)
    except Exception as e:
        print(f"Erreur sur données complètes : {e}")
        labels_hdbscan = np.full(len(X_clust), -1)

    n_cl_hdb  = len(set(labels_hdbscan)) - (1 if -1 in labels_hdbscan else 0)
    noise_hdb = (labels_hdbscan == -1).sum()
    print(f"  → {n_cl_hdb} clusters, {noise_hdb} bruit ({noise_hdb/len(labels_hdbscan)*100:.1f}%)")

    metrics_all['HDBSCAN']      = compute_metrics(X_clust, labels_hdbscan, labels_final)
    all_clusterings['HDBSCAN']  = labels_hdbscan

    # --- Visualisation ---
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # a. Grid search heatmap
    mcs_list = sorted(df_hdb_grid['min_cluster_size'].unique())
    ms_list  = df_hdb_grid['min_samples'].unique().tolist()
    pivot_n  = df_hdb_grid.pivot(index='min_samples', columns='min_cluster_size', values='n_clusters').fillna(0)
    im = axes[0].imshow(pivot_n.values, aspect='auto', cmap='YlOrRd')
    axes[0].set_xticks(range(len(mcs_list)))
    axes[0].set_xticklabels(mcs_list, fontsize=9)
    axes[0].set_yticks(range(len(pivot_n.index)))
    axes[0].set_yticklabels(pivot_n.index, fontsize=9)
    axes[0].set_xlabel("min_cluster_size", fontsize=11)
    axes[0].set_ylabel("min_samples", fontsize=11)
    for (i, j), val in np.ndenumerate(pivot_n.values):
        axes[0].text(j, i, f'{int(val)}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=axes[0], label='n_clusters')
    axes[0].set_title("HDBSCAN — Grid search n_clusters", fontsize=12)

    # b. Clusters PC1-PC2
    for lbl in sorted(set(labels_hdbscan)):
        mask = labels_hdbscan == lbl
        if lbl == -1:
            axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
        else:
            axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                           s=15, alpha=0.6, label=f'C{lbl+1}')
    axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
    axes[1].set_title(f"HDBSCAN — {n_cl_hdb} clusters (mcs={BEST_MCS})", fontsize=13)
    if n_cl_hdb <= 10:
        axes[1].legend(fontsize=9, markerscale=2)
    axes[1].grid(alpha=0.2)

    # c. Profil biologique
    if n_cl_hdb >= 2:
        df_hdb_bio = df_hellinger.copy()
        df_hdb_bio['cluster'] = labels_hdbscan
        df_hdb_bio = df_hdb_bio[df_hdb_bio['cluster'] != -1]
        bio_hdb = df_hdb_bio.groupby('cluster')[COLS_CONC].mean()
        im2 = axes[2].imshow(bio_hdb.values.T, aspect='auto', cmap='YlOrRd')
        axes[2].set_xticks(range(len(bio_hdb)))
        axes[2].set_xticklabels([f'C{i+1}' for i in bio_hdb.index], fontsize=10)
        axes[2].set_yticks(range(len(COLS_CONC)))
        axes[2].set_yticklabels(COLS_CONC, fontsize=8)
        plt.colorbar(im2, ax=axes[2], label='Hellinger (moy.)')
        axes[2].set_title("Profil biologique HDBSCAN", fontsize=13)

    plt.tight_layout()
    plt.savefig('fig_21_hdbscan.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f"\nMétriques HDBSCAN :")
    print(f"  Silhouette     : {metrics_all['HDBSCAN']['silhouette']:.4f}")
    print(f"  Davies-Bouldin : {metrics_all['HDBSCAN']['davies_bouldin']:.4f}")
    print(f"  Calinski-Harab.: {metrics_all['HDBSCAN']['calinski_harabasz']:.1f}")
    print(f"  ARI vs K-means : {metrics_all['HDBSCAN']['ari']:.4f}")
else:
    print("HDBSCAN non disponible.")
    labels_hdbscan = np.full(len(X_clust), -1)
    metrics_all['HDBSCAN']     = compute_metrics(X_clust, labels_hdbscan, labels_final)
    all_clusterings['HDBSCAN'] = labels_hdbscan

## 4.3 OPTICS : Ordering Points to Identify the Clustering Structure (Ankerst et al., 1999)

OPTICS généralise DBSCAN en ordonnant les points selon leur *reachability distance*, ce qui permet de détecter des **clusters de densités variables**. Le **reachability plot** est la signature visuelle de la structure hiérarchique des données.

**Paramètres clés :**
- `min_samples` : nombre minimum de points dans le voisinage (analogue à DBSCAN)
- `max_eps` : rayon maximum de recherche (∞ par défaut)

**Avantages :** multi-densité, pas d'ε à fixer, reachability plot très informatif  
**Limites :** O(n log n), extraction des clusters non triviale, sous-échantillonnage requis ici

In [ ]:
# === OPTICS ===

N_OPT    = min(4000, len(X_clust))
rng_opt  = np.random.default_rng(42)
idx_opt  = rng_opt.choice(len(X_clust), N_OPT, replace=False)
X_opt    = X_clust[idx_opt]

# Grid search min_samples
ms_optics   = [5, 10, 20, 30]
opt_results = []
print("Grid search OPTICS (min_samples)...")
for ms in ms_optics:
    opt     = OPTICS(min_samples=ms, metric='euclidean', n_jobs=-1)
    opt.fit(X_opt)
    labs_xi = opt.labels_
    n_cl_xi = len(set(labs_xi)) - (1 if -1 in labs_xi else 0)
    noise_xi = (labs_xi == -1).sum() / len(labs_xi) * 100
    if n_cl_xi >= 2 and (labs_xi != -1).sum() > n_cl_xi:
        sil_xi = silhouette_score(X_opt[labs_xi != -1], labs_xi[labs_xi != -1],
                                  sample_size=min(1000, (labs_xi != -1).sum()), random_state=42)
    else:
        sil_xi = np.nan
    opt_results.append({'min_samples': ms, 'n_clusters': n_cl_xi,
                        'noise_pct': round(noise_xi, 1),
                        'silhouette': round(sil_xi, 4) if not np.isnan(sil_xi) else np.nan})
    print(f"  ms={ms:2d} → {n_cl_xi} clusters, {noise_xi:.1f}% bruit")

df_optics_grid = pd.DataFrame(opt_results)
valid_opt  = df_optics_grid[(df_optics_grid['n_clusters'] >= 2) &
                             (df_optics_grid['noise_pct'] < 50) &
                             (df_optics_grid['silhouette'].notna())]
BEST_MS_OPT = int(valid_opt.loc[valid_opt['silhouette'].idxmax(), 'min_samples']) if len(valid_opt) else 10
print(f"\nMeilleur min_samples OPTICS : {BEST_MS_OPT}")

# Application avec meilleur paramètre (même sous-échantillon)
optics_final    = OPTICS(min_samples=BEST_MS_OPT, metric='euclidean', n_jobs=-1)
optics_final.fit(X_opt)
labels_optics_sub = optics_final.labels_

# Extension aux données complètes
if N_OPT < len(X_clust):
    from sklearn.neighbors import KNeighborsClassifier as _KNN
    _knn_opt = _KNN(n_neighbors=5)
    _knn_opt.fit(X_opt, labels_optics_sub)
    labels_optics = _knn_opt.predict(X_clust)
else:
    labels_optics = labels_optics_sub

n_cl_opt  = len(set(labels_optics)) - (1 if -1 in labels_optics else 0)
noise_opt = (labels_optics == -1).sum()
print(f"  → {n_cl_opt} clusters, {noise_opt} bruit ({noise_opt/len(labels_optics)*100:.1f}%)")

metrics_all['OPTICS']      = compute_metrics(X_clust, labels_optics, labels_final)
all_clusterings['OPTICS']  = labels_optics

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. Reachability plot
order           = optics_final.ordering_
reach           = optics_final.reachability_[order]
labs_ordered    = labels_optics_sub[order]
unique_opt_labs = sorted(set(labels_optics_sub[labels_optics_sub != -1]))

for lbl in unique_opt_labs:
    mask_r = labs_ordered == lbl
    axes[0].vlines(np.where(mask_r)[0], 0, reach[mask_r],
                   colors=[plt.cm.tab10(lbl % 10)], alpha=0.7, label=f'C{lbl+1}')
mask_noise_r = labs_ordered == -1
axes[0].vlines(np.where(mask_noise_r)[0], 0, reach[mask_noise_r], colors='lightgray', alpha=0.3)
axes[0].set_xlabel("Points (ordonnés)", fontsize=11)
axes[0].set_ylabel("Reachability distance", fontsize=11)
axes[0].set_title(f"OPTICS — Reachability plot (ms={BEST_MS_OPT})", fontsize=12)
if len(unique_opt_labs) <= 8:
    axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)

# b. Clusters PC1-PC2
for lbl in sorted(set(labels_optics)):
    mask = labels_optics == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"OPTICS — {n_cl_opt} clusters (ms={BEST_MS_OPT})", fontsize=13)
if n_cl_opt <= 10:
    axes[1].legend(fontsize=9, markerscale=2)
axes[1].grid(alpha=0.2)

# c. Profil biologique
if n_cl_opt >= 2:
    df_opt_bio = df_hellinger.copy()
    df_opt_bio['cluster'] = labels_optics
    df_opt_bio = df_opt_bio[df_opt_bio['cluster'] != -1]
    bio_opt = df_opt_bio.groupby('cluster')[COLS_CONC].mean()
    im3 = axes[2].imshow(bio_opt.values.T, aspect='auto', cmap='YlOrRd')
    axes[2].set_xticks(range(len(bio_opt)))
    axes[2].set_xticklabels([f'C{i+1}' for i in bio_opt.index], fontsize=10)
    axes[2].set_yticks(range(len(COLS_CONC)))
    axes[2].set_yticklabels(COLS_CONC, fontsize=8)
    plt.colorbar(im3, ax=axes[2], label='Hellinger (moy.)')
    axes[2].set_title("Profil biologique OPTICS", fontsize=13)

plt.tight_layout()
plt.savefig('fig_22_optics.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques OPTICS :")
print(f"  Silhouette     : {metrics_all['OPTICS']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['OPTICS']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['OPTICS']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['OPTICS']['ari']:.4f}")

## 4.4 Spectral Clustering (Shi & Malik, 2000)

Le Spectral Clustering utilise la **théorie des graphes spectraux** pour détecter des structures non-convexes. Il construit un graphe de similarité entre points (k-NN ou RBF), calcule le Laplacien normalisé, puis applique K-means dans l'espace des vecteurs propres.

**Paramètres clés :**
- `n_clusters` : nombre de clusters (optimisé par silhouette)
- `affinity` : type de graphe (`nearest_neighbors` utilisé ici pour la scalabilité)

**Avantages :** capture les structures non-convexes, robuste aux géométries complexes  
**Limites :** O(n²) en mémoire, nécessite k, peu scalable → sous-échantillonnage nécessaire

In [ ]:
# === Spectral Clustering ===

# Sous-échantillon (matrice d'affinité O(n²))
N_SPEC   = min(3000, len(X_clust))
rng_spec = np.random.default_rng(42)
idx_spec = rng_spec.choice(len(X_clust), N_SPEC, replace=False)
X_spec   = X_clust[idx_spec]

# Grid search k via silhouette
k_vals_spec  = range(2, 10)
spec_results = []
print("Spectral Clustering — optimisation k (sous-échantillon)...")
for k in k_vals_spec:
    sc_tmp = SpectralClustering(n_clusters=k, affinity='nearest_neighbors',
                                n_neighbors=10, random_state=42, n_jobs=-1)
    labs_sc = sc_tmp.fit_predict(X_spec)
    sil_sc  = silhouette_score(X_spec, labs_sc,
                               sample_size=min(1000, len(X_spec)), random_state=42)
    spec_results.append({'k': k, 'silhouette': sil_sc})
    print(f"  k={k} → silhouette={sil_sc:.4f}")

df_spec_grid = pd.DataFrame(spec_results)
best_k_spec  = int(df_spec_grid.loc[df_spec_grid['silhouette'].idxmax(), 'k'])
print(f"\nk optimal Spectral : {best_k_spec}")

# Application avec k optimal sur le sous-échantillon
sc_final       = SpectralClustering(n_clusters=best_k_spec, affinity='nearest_neighbors',
                                    n_neighbors=10, random_state=42, n_jobs=-1)
labels_spec_sub = sc_final.fit_predict(X_spec)

# Extension aux données complètes par KNN
from sklearn.neighbors import KNeighborsClassifier
knn_ext = KNeighborsClassifier(n_neighbors=5)
knn_ext.fit(X_spec, labels_spec_sub)
labels_spectral = knn_ext.predict(X_clust)
n_cl_spec       = len(np.unique(labels_spectral))

metrics_all['Spectral']     = compute_metrics(X_clust, labels_spectral, labels_final)
all_clusterings['Spectral'] = labels_spectral

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. Silhouette vs k
axes[0].plot(df_spec_grid['k'], df_spec_grid['silhouette'], 'o-', color='steelblue', lw=2)
axes[0].axvline(best_k_spec, color='crimson', linestyle='--', label=f'k optimal={best_k_spec}')
axes[0].set_xlabel("Nombre de clusters k", fontsize=12)
axes[0].set_ylabel("Score de silhouette", fontsize=12)
axes[0].set_title("Spectral — Silhouette vs k", fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

# b. Clusters PC1-PC2
for lbl in sorted(np.unique(labels_spectral)):
    mask = labels_spectral == lbl
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[plt.cm.Set1(lbl / max(n_cl_spec - 1, 1))],
                    s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"Spectral Clustering — {n_cl_spec} clusters", fontsize=13)
axes[1].legend(fontsize=9, markerscale=2); axes[1].grid(alpha=0.2)

# c. Profil biologique
df_spec_bio = df_hellinger.copy()
df_spec_bio['cluster'] = labels_spectral
bio_spec = df_spec_bio.groupby('cluster')[COLS_CONC].mean()
im4 = axes[2].imshow(bio_spec.values.T, aspect='auto', cmap='YlOrRd')
axes[2].set_xticks(range(len(bio_spec)))
axes[2].set_xticklabels([f'C{i+1}' for i in bio_spec.index], fontsize=10)
axes[2].set_yticks(range(len(COLS_CONC)))
axes[2].set_yticklabels(COLS_CONC, fontsize=8)
plt.colorbar(im4, ax=axes[2], label='Hellinger (moy.)')
axes[2].set_title("Profil biologique Spectral", fontsize=13)

plt.tight_layout()
plt.savefig('fig_23_spectral.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques Spectral :")
print(f"  Silhouette     : {metrics_all['Spectral']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['Spectral']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['Spectral']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['Spectral']['ari']:.4f}")

## 4.5 Gaussian Mixture Models : GMM (Fraley & Raftery, 2002)

Les GMM modélisent les données comme un **mélange de distributions gaussiennes**. Contrairement aux méthodes dures (K-means, DBSCAN), ils produisent une **assignation probabiliste** : chaque point reçoit une probabilité d'appartenance à chaque composante.

**Sélection du modèle :**
- **BIC** (Bayesian Information Criterion) : pénalise la complexité → favorise les modèles parcimonieux
- **AIC** (Akaike Information Criterion) : moins pénalisant, favorise l'ajustement

**Structures de covariance testées :** `full`, `tied`, `diag`, `spherical`

**Avantages :** doux, probabiliste, flexibilité de la covariance, sélection automatique via BIC/AIC  
**Limites :** suppose la normalité des composantes, nécessite k, sensible à l'initialisation

In [ ]:
# === GMM — Gaussian Mixture Models ===

k_vals_gmm = range(2, 11)
cov_types   = ['full', 'tied', 'diag', 'spherical']
gmm_results = []

print("GMM — sélection via BIC/AIC (grille k × covariance)...")
for k in k_vals_gmm:
    for cov in cov_types:
        try:
            gmm = GaussianMixture(n_components=k, covariance_type=cov,
                                  random_state=42, max_iter=200, n_init=3)
            gmm.fit(X_clust)
            bic  = gmm.bic(X_clust)
            aic  = gmm.aic(X_clust)
            labs = gmm.predict(X_clust)
            n_cl = len(np.unique(labs))
            sil  = silhouette_score(X_clust, labs,
                                    sample_size=min(3000, len(X_clust)), random_state=42) if n_cl >= 2 else np.nan
            gmm_results.append({'k': k, 'covariance_type': cov,
                                 'BIC': bic, 'AIC': aic,
                                 'silhouette': sil, 'converged': gmm.converged_})
        except Exception:
            gmm_results.append({'k': k, 'covariance_type': cov,
                                 'BIC': np.nan, 'AIC': np.nan,
                                 'silhouette': np.nan, 'converged': False})

df_gmm = pd.DataFrame(gmm_results)
df_gmm_ok = df_gmm[df_gmm['converged'] & df_gmm['BIC'].notna()]

# Sélection : min BIC
best_gmm_row = df_gmm_ok.loc[df_gmm_ok['BIC'].idxmin()]
BEST_K_GMM   = int(best_gmm_row['k'])
BEST_COV_GMM = best_gmm_row['covariance_type']
print(f"Meilleur modèle GMM : k={BEST_K_GMM}, covariance='{BEST_COV_GMM}'")
print(f"  BIC={best_gmm_row['BIC']:.1f}, AIC={best_gmm_row['AIC']:.1f}")

# Application finale
gmm_final = GaussianMixture(n_components=BEST_K_GMM, covariance_type=BEST_COV_GMM,
                             random_state=42, max_iter=200, n_init=5)
gmm_final.fit(X_clust)
labels_gmm = gmm_final.predict(X_clust)
proba_gmm  = gmm_final.predict_proba(X_clust)
max_proba  = proba_gmm.max(axis=1)

metrics_all['GMM']        = compute_metrics(X_clust, labels_gmm, labels_final)
all_clusterings['GMM']    = labels_gmm

# --- Visualisation ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

colors_cov = {'full': '#e41a1c', 'tied': '#377eb8', 'diag': '#4daf4a', 'spherical': '#ff7f00'}

# a. BIC
for cov in cov_types:
    sub = df_gmm_ok[df_gmm_ok['covariance_type'] == cov].sort_values('k')
    if len(sub):
        axes[0, 0].plot(sub['k'], sub['BIC'], 'o-', label=cov, color=colors_cov[cov])
axes[0, 0].axvline(BEST_K_GMM, color='black', linestyle=':', alpha=0.7)
axes[0, 0].set_xlabel("k (n_components)", fontsize=12)
axes[0, 0].set_ylabel("BIC", fontsize=12)
axes[0, 0].set_title("GMM — Sélection via BIC", fontsize=13)
axes[0, 0].legend(fontsize=10); axes[0, 0].grid(alpha=0.3)

# b. AIC
for cov in cov_types:
    sub = df_gmm_ok[df_gmm_ok['covariance_type'] == cov].sort_values('k')
    if len(sub):
        axes[0, 1].plot(sub['k'], sub['AIC'], 'o-', label=cov, color=colors_cov[cov])
axes[0, 1].axvline(BEST_K_GMM, color='black', linestyle=':', alpha=0.7)
axes[0, 1].set_xlabel("k (n_components)", fontsize=12)
axes[0, 1].set_ylabel("AIC", fontsize=12)
axes[0, 1].set_title("GMM — Sélection via AIC", fontsize=13)
axes[0, 1].legend(fontsize=10); axes[0, 1].grid(alpha=0.3)

# c. Clusters PC1-PC2
for lbl in sorted(np.unique(labels_gmm)):
    mask = labels_gmm == lbl
    axes[1, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                       c=[plt.cm.Set1(lbl / max(BEST_K_GMM - 1, 1))],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1, 0].set_xlabel("PC1", fontsize=12); axes[1, 0].set_ylabel("PC2", fontsize=12)
axes[1, 0].set_title(f"GMM — {BEST_K_GMM} composantes ({BEST_COV_GMM})", fontsize=13)
axes[1, 0].legend(fontsize=9, markerscale=2); axes[1, 0].grid(alpha=0.2)

# d. Incertitude d'assignation
sc = axes[1, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=max_proba,
                         cmap='RdYlGn', s=10, alpha=0.6, vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=axes[1, 1], label='P(meilleur cluster)')
axes[1, 1].set_xlabel("PC1", fontsize=12); axes[1, 1].set_ylabel("PC2", fontsize=12)
axes[1, 1].set_title("GMM — Incertitude d'assignation", fontsize=13)
axes[1, 1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_24_gmm.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques GMM :")
print(f"  Silhouette     : {metrics_all['GMM']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['GMM']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['GMM']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['GMM']['ari']:.4f}")
print(f"  Incertitude moy. (1-P): {(1 - max_proba).mean():.4f}")

## 4.6 Comparaison Globale des Clusterings

Cette section compare systématiquement tous les algorithmes selon trois axes :
- **Métriques internes** : Silhouette (↑), Davies-Bouldin (↓), Calinski-Harabasz (↑)
- **Métriques externes** vs K-means baseline : ARI, NMI, V-measure (valeurs entre 0 et 1)
- **Structure visuelle** : projection dans le plan PC1–PC2

In [ ]:
# === Tableau comparatif global des métriques ===

methods_order_cmp = ['K-means', 'DBSCAN', 'HDBSCAN', 'OPTICS', 'Spectral', 'GMM']
rows_cmp = []
for method in methods_order_cmp:
    if method not in metrics_all:
        continue
    m = metrics_all[method]
    noise_n = m['n_noise'] if method != 'K-means' else 0
    noise_pct = f"{noise_n/len(X_clust)*100:.1f}%" if noise_n > 0 else "0%"
    rows_cmp.append({
        'Méthode'             : method,
        'k'                   : m['n_clusters'],
        'Bruit (%)'           : noise_pct,
        'Silhouette ↑'        : f"{m['silhouette']:.4f}"        if not np.isnan(m['silhouette'])        else '—',
        'Davies-Bouldin ↓'    : f"{m['davies_bouldin']:.4f}"    if not np.isnan(m['davies_bouldin'])    else '—',
        'Calinski-Harabasz ↑' : f"{m['calinski_harabasz']:.1f}" if not np.isnan(m['calinski_harabasz']) else '—',
        'ARI vs baseline'     : f"{m['ari']:.4f}"               if not np.isnan(m['ari'])               else '—',
        'NMI vs baseline'     : f"{m['nmi']:.4f}"               if not np.isnan(m['nmi'])               else '—',
        'V-measure'           : f"{m['v_measure']:.4f}"         if not np.isnan(m['v_measure'])         else '—',
    })

df_compare = pd.DataFrame(rows_cmp)
print("=" * 90)
print("TABLEAU COMPARATIF — Métriques de clustering (Jalon 3)")
print("=" * 90)
print(df_compare.to_string(index=False))
print("\nLégende :")
print("  ↑ = plus grand est meilleur | ↓ = plus petit est meilleur")
print("  ARI/NMI/V-measure : accord avec le K-means baseline (1=identique, 0=indépendant)")
print("  Bruit : points non assignés (propre aux méthodes de densité)")

In [ ]:
# === Visualisation comparative globale ===

fig, axes = plt.subplots(2, 3, figsize=(21, 12))

methods_order_plot = ['K-means', 'DBSCAN', 'HDBSCAN', 'OPTICS', 'Spectral', 'GMM']
methods_avail = [m for m in methods_order_plot if m in metrics_all]

# --- a. Heatmap métriques internes (normalisées) ---
sil_vals  = [metrics_all[m]['silhouette']         for m in methods_avail]
db_vals   = [metrics_all[m]['davies_bouldin']      for m in methods_avail]
ch_vals   = [metrics_all[m]['calinski_harabasz']   for m in methods_avail]
db_inv    = [1/v if (not np.isnan(v) and v > 0) else np.nan for v in db_vals]
metrics_matrix = np.array([sil_vals, db_inv, ch_vals]).T

metrics_norm = np.zeros_like(metrics_matrix)
for j in range(metrics_matrix.shape[1]):
    col = metrics_matrix[:, j]
    valid = ~np.isnan(col)
    if valid.sum() > 0:
        cmin, cmax = col[valid].min(), col[valid].max()
        if cmax > cmin:
            metrics_norm[:, j] = np.where(valid, (col - cmin) / (cmax - cmin), 0)
        else:
            metrics_norm[:, j] = np.where(valid, 0.5, 0)

im = axes[0, 0].imshow(metrics_norm, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
axes[0, 0].set_xticks([0, 1, 2])
axes[0, 0].set_xticklabels(['Silhouette ↑', '1/DB ↑', 'CH ↑'], fontsize=10)
axes[0, 0].set_yticks(range(len(methods_avail)))
axes[0, 0].set_yticklabels(methods_avail, fontsize=11)
for (i, j), nval in np.ndenumerate(metrics_norm):
    raw = metrics_matrix[i, j]
    txt = f'{raw:.3f}' if not np.isnan(raw) else '—'
    axes[0, 0].text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='black' if nval < 0.7 else 'white')
axes[0, 0].set_title("Métriques internes (normalisées [0–1])", fontsize=12)

# --- b. Heatmap métriques externes vs K-means ---
ext_matrix = np.array([
    [metrics_all[m]['ari'], metrics_all[m]['nmi'], metrics_all[m]['v_measure']]
    for m in methods_avail
])
im2 = axes[0, 1].imshow(np.nan_to_num(ext_matrix, nan=0), aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 1].set_xticks([0, 1, 2])
axes[0, 1].set_xticklabels(['ARI', 'NMI', 'V-measure'], fontsize=10)
axes[0, 1].set_yticks(range(len(methods_avail)))
axes[0, 1].set_yticklabels(methods_avail, fontsize=11)
for (i, j), val in np.ndenumerate(ext_matrix):
    txt = f'{val:.3f}' if not np.isnan(val) else '—'
    axes[0, 1].text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='black' if (np.isnan(val) or val < 0.6) else 'white')
axes[0, 1].set_title("Métriques externes vs K-means baseline", fontsize=12)

# --- c. Nombre de clusters ---
k_vals_bar = [metrics_all[m]['n_clusters'] for m in methods_avail]
colors_bar = ['#4393c3', '#d6604d', '#74add1', '#f4a582', '#fdae61', '#a6d96a']
bars = axes[0, 2].barh(methods_avail, k_vals_bar, color=colors_bar[:len(methods_avail)])
axes[0, 2].set_xlabel("Nombre de clusters", fontsize=11)
axes[0, 2].set_title("Nombre de clusters par méthode", fontsize=12)
for i, v in enumerate(k_vals_bar):
    axes[0, 2].text(v + 0.1, i, str(v), va='center', fontsize=10)
axes[0, 2].grid(axis='x', alpha=0.3)

# --- d/e/f. Clusters PC1-PC2 pour K-means, DBSCAN et GMM ---
subplot_methods = [m for m in ['K-means', 'DBSCAN', 'GMM'] if m in all_clusterings]
subplot_positions = [(1, 0), (1, 1), (1, 2)]
for (row, col), method in zip(subplot_positions, subplot_methods):
    labs = all_clusterings[method]
    unique_l = sorted(set(labs))
    for lbl in unique_l:
        mask = labs == lbl
        if lbl == -1:
            axes[row, col].scatter(X_pca[mask, 0], X_pca[mask, 1],
                                  c='lightgray', s=5, alpha=0.2, label='Bruit')
        else:
            axes[row, col].scatter(X_pca[mask, 0], X_pca[mask, 1],
                                  c=[plt.cm.tab10(lbl % 10)], s=10, alpha=0.5,
                                  label=f'C{lbl+1}')
    axes[row, col].set_xlabel("PC1", fontsize=10)
    axes[row, col].set_ylabel("PC2", fontsize=10)
    n_cl = metrics_all[method]['n_clusters']
    axes[row, col].set_title(f"{method} — {n_cl} clusters", fontsize=12)
    if n_cl <= 8:
        axes[row, col].legend(fontsize=8, markerscale=2)
    axes[row, col].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_25_comparaison_globale.png', dpi=120, bbox_inches='tight')
plt.show()

---

# Partie 4 : Conclusions Générales

## 5.1 Synthèse des Résultats

### Ce que nous avons appris

Cette étude a appliqué une démarche complète d'apprentissage non supervisé sur les concentrations de plancton marin issues du jeu de données SEANOE (Panaïotis et al., 2023).

**Jalon 1 : Baseline :** La transformation de Hellinger suivie d'une ACP et d'un K-means a permis de reproduire les grandes lignes de l'analyse de référence. Le score de silhouette a guidé la sélection du nombre optimal de clusters, révélant des communautés planctoniques distinctes associées à des conditions environnementales contrastées (couche, température, chlorophylle).

**Jalon 2 : Représentations alternatives :** L'AFC, l'Isomap, le LLE et le MDS ont offert des perspectives complémentaires sur la structure des données. Les méthodes non-linéaires (Isomap, LLE) ont révélé des structures curvilinéaires invisibles avec l'ACP, tandis que l'AFC a mis en évidence les associations entre groupes taxonomiques.

**Jalon 3 : Clusterings alternatifs :** Les cinq algorithmes explorés (K-means, DBSCAN, HDBSCAN, OPTICS, Spectral, GMM) ont chacun révélé différentes facettes de la structure des communautés :
- Les algorithmes de **densité** (DBSCAN, HDBSCAN, OPTICS) ont identifié des points de bruit potentiellement associés à des observations rares ou à des conditions extrêmes
- Le **Spectral Clustering** a capturé des structures non-convexes dans l'espace PCA
- les **GMM** ont fourni une assignation probabiliste permettant de quantifier l'incertitude taxonomique

### Apports Écologiques

Les communautés planctoniques identifiées reflètent les grandes provinces biogéographiques océaniques, avec une structuration forte par :
1. La **profondeur** (couche épipélagique vs méso- et bathypélagique)
2. La **productivité** (zones oligotrophes vs eutrophes)
3. Les **saisons** et la variabilité temporelle (2008–2019)

### Limites et Perspectives

- L'utilisation d'un sous-échantillon pour OPTICS et Spectral Clustering peut introduire un biais
- Les méthodes de densité sont sensibles à la dimensionnalité de l'espace PCA
- Des analyses complémentaires pourraient intégrer les variables environnementales directement dans le clustering (co-inertie, RDA)

## 5.2 Références Bibliographiques

- **Panaïotis, T. et al. (2023).** Content-aware segmentation of objects spanning a large size range. *Frontiers in Marine Science*, 10, 1280510.
- **Hellinger, E. (1909).** Neue Begründung der Theorie quadratischer Formen von unendlichvielen Veränderlichen. *Journal für die reine und angewandte Mathematik*, 136, 210–271.
- **Pearson, K. (1901).** On Lines and Planes of Closest Fit to Systems of Points in Space. *Philosophical Magazine*, 2(11), 559–572.
- **Benzécri, J.-P. (1973).** *L'Analyse des Données. Vol. 2: L'Analyse des Correspondances.* Dunod, Paris.
- **Tenenbaum, J. B., de Silva, V., & Langford, J. C. (2000).** A global geometric framework for nonlinear dimensionality reduction. *Science*, 290(5500), 2319–2323.
- **Roweis, S. T., & Saul, L. K. (2000).** Nonlinear dimensionality reduction by locally linear embedding. *Science*, 290(5500), 2323–2326.
- **Kruskal, J. B. (1964).** Multidimensional scaling by optimizing goodness of fit to a nonmetric hypothesis. *Psychometrika*, 29(1), 1–27.
- **Ester, M. et al. (1996).** A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD*, 96(34), 226–231.
- **Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013).** Density-based clustering based on hierarchical density estimates. *PAKDD*.
- **Ankerst, M. et al. (1999).** OPTICS: Ordering points to identify the clustering structure. *ACM SIGMOD*, 28(2), 49–60.
- **Shi, J., & Malik, J. (2000).** Normalized cuts and image segmentation. *IEEE TPAMI*, 22(8), 888–905.
- **Fraley, C., & Raftery, A. E. (2002).** Model-based clustering, discriminant analysis, and density estimation. *JASA*, 97(458), 611–631.